# FGAI (ICT4403) Assignment 3 - English→Hindi MT with **small models** (all-in-one notebook)

**Basic Transformer (scratch) · mT5-small · Llama-3.2-1B-Instruct (LoRA) · Gemma-3-1B-it (LoRA)** - compared on BLEU, chrF, parameters, training/inference time, sampling temperature and responsible-AI probes.
Nothing to upload: this notebook writes all the project files itself (cells starting with `%%writefile`), then runs them.

**Kaggle setup (once):** Settings → Accelerator **GPU T4 x2** · Internet **On** · Add-ons → Secrets → `HF_TOKEN` (attached to this notebook).
Works on Colab too (T4 GPU; add `HF_TOKEN` under the 🔑 Secrets panel).

**How to run:** 1) leave `TINY_RUN = True` and *Run All* (a few minutes - checks everything). 2) set `TINY_RUN = False`, then **Save Version → Save & Run All (Commit)** on Kaggle (runs in the background). 3) download `results_bundle.zip` and `project_for_github.zip` from the Output tab.

## 1. Settings

In [ ]:
TINY_RUN = True            # True = tiny pipeline check (minutes). False = the real experiment.
LIGHT = True               # small training subsets so everything fits comfortably on a T4
CLEAN_CHECKPOINTS = True   # delete big model weights at the end (keeps the output small)

LLAMA_ID = "meta-llama/Llama-3.2-1B-Instruct"   # smallest Llama 3.x
GEMMA_ID = "google/gemma-3-1b-it"               # even smaller option: "google/gemma-3-270m" (accept its licence on Hugging Face first)

LIGHT_SETTINGS = {
    "N_TRAIN_SCRATCH": "50000", "SCRATCH_EPOCHS": "10",
    "N_TRAIN_MT5": "20000",     "MT5_EPOCHS": "2",
    "N_TRAIN_LLM": "5000",
    "N_TEMP_TEST": "100",
}
TINY = dict(POOL_SIZE="2000", N_TEST="50", N_TEMP_TEST="20", N_TRAIN_SCRATCH="2000", N_TRAIN_MT5="1000",
            N_TRAIN_LLM="500", SCRATCH_EPOCHS="2", MT5_EPOCHS="1", SCRATCH_VOCAB="4000")

OVERRIDES = {"LLAMA_ID": LLAMA_ID, "GEMMA_ID": GEMMA_ID}
if LIGHT:
    OVERRIDES.update(LIGHT_SETTINGS)
# Still too slow? lower sizes here, e.g.  OVERRIDES["N_TRAIN_MT5"] = "10000"

## 2. Project folder and GPU check

In [ ]:
import os, shutil, subprocess, sys, zipfile
from pathlib import Path
import torch

if Path("/kaggle/working").exists():
    ROOT, PLATFORM = Path("/kaggle/working"), "kaggle"
elif Path("/content").exists():
    ROOT, PLATFORM = Path("/content"), "colab"
else:
    ROOT, PLATFORM = Path.cwd(), "local"

PROJECT = ROOT / ("en-hi-mt-tiny" if TINY_RUN else "en-hi-mt-comparison")   # the tiny test never mixes with real results
for d in ["src", "results", "data", "checkpoints", "notebooks"]:
    (PROJECT / d).mkdir(parents=True, exist_ok=True)
os.chdir(PROJECT)

assert torch.cuda.is_available(), "No GPU! Kaggle: Settings -> Accelerator -> GPU T4 x2.  Colab: Runtime -> Change runtime type -> T4 GPU."
print("platform:", PLATFORM, "| project dir:", PROJECT)
print("GPU:", torch.cuda.get_device_name(0), "| TINY_RUN =", TINY_RUN, "| LIGHT =", LIGHT)

## 3. Project files (each cell below writes one file - this is the code you will push to GitHub)

In [ ]:
%%writefile requirements.txt
# Core
torch>=2.3
transformers>=4.46,<5.0
peft>=0.13
datasets>=2.20
sentencepiece>=0.2.0
protobuf>=4.25
sacrebleu>=2.4
# Reporting
pandas>=2.0
matplotlib>=3.7
packaging
# Optional: 4-bit loading of bigger models (--load_4bit)
# bitsandbytes>=0.43


In [ ]:
%%writefile src/__init__.py
"""EN->HI MT comparison package."""
import os

# Must be set before PyTorch touches the GPU / before transformers is imported.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")   # less memory fragmentation
os.environ.setdefault("USE_TF", "0")                                           # Colab ships TensorFlow; we never need it
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")


In [ ]:
%%writefile src/config.py
"""Central configuration for the English -> Hindi MT comparison.

Every value can be overridden with an environment variable of the same name,
e.g.  N_TRAIN_SCRATCH=20000 python -m src.train_transformer
"""
import os
from pathlib import Path


def _env(name, default):
    """Read an env var and cast it to the type of `default`."""
    raw = os.environ.get(name)
    if raw is None:
        return default
    return type(default)(raw)


# --------------------------------------------------------------------------- #
# Paths
# --------------------------------------------------------------------------- #
ROOT = Path(__file__).resolve().parent.parent
DATA_DIR = ROOT / "data"
RESULTS_DIR = ROOT / "results"
CKPT_DIR = ROOT / "checkpoints"
for _d in (DATA_DIR, RESULTS_DIR, CKPT_DIR):
    _d.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------------------------------- #
# Language pair + dataset
# --------------------------------------------------------------------------- #
SRC_LANG, TGT_LANG = "en", "hi"
SRC_NAME, TGT_NAME = "English", "Hindi"
DATASET_ID = "cfilt/iitb-english-hindi"          # IIT Bombay English-Hindi corpus
SEED = _env("SEED", 42)

# Size of the cleaned training pool that all models draw from (nested subsets).
POOL_SIZE = _env("POOL_SIZE", 100_000)
N_TEST = _env("N_TEST", 500)                      # sentences used for the main comparison
N_TEMP_TEST = _env("N_TEMP_TEST", 200)            # subset used for the temperature sweep
MAX_WORDS = _env("MAX_WORDS", 50)                 # max words per side (train)
MIN_WORDS = _env("MIN_WORDS", 3)

# --------------------------------------------------------------------------- #
# Decoding / evaluation (identical for every model)
# --------------------------------------------------------------------------- #
TEMPERATURES = [0.0, 0.3, 0.7, 1.0]               # 0.0 == greedy decoding
EVAL_BATCH_SIZE = _env("EVAL_BATCH_SIZE", 32)
MAX_NEW_TOKENS = _env("MAX_NEW_TOKENS", 160)

# --------------------------------------------------------------------------- #
# Model 1: Transformer trained from scratch
# --------------------------------------------------------------------------- #
N_TRAIN_SCRATCH = _env("N_TRAIN_SCRATCH", 100_000)
SCRATCH = dict(
    vocab_size=_env("SCRATCH_VOCAB", 16_000),
    d_model=256, n_heads=4, n_enc_layers=3, n_dec_layers=3, d_ff=1024, dropout=0.1,
    max_tokens=100,                               # drop training pairs longer than this (subword tokens)
    batch_size=64, epochs=_env("SCRATCH_EPOCHS", 12), lr=7e-4, warmup=1000,
    label_smoothing=0.1, patience=3,
)

# --------------------------------------------------------------------------- #
# Model 2: mT5 (fine-tuned)
# --------------------------------------------------------------------------- #
N_TRAIN_MT5 = _env("N_TRAIN_MT5", 50_000)
MT5 = dict(
    model_id=_env("MT5_ID", "google/mt5-small"),
    prefix="translate English to Hindi: ",
    max_src=64, max_tgt=96,
    # mT5 has a 250k-token vocabulary: logits for batch x length x 250k floats dominate GPU memory,
    # so use a small micro-batch and accumulate gradients (effective batch = 8 x 4 = 32).
    batch_size=8, grad_accum=4, epochs=_env("MT5_EPOCHS", 2), lr=5e-4, warmup=200,
)

# --------------------------------------------------------------------------- #
# Models 3 & 4: decoder-only LLMs (LLaMA, Gemma) -- LoRA fine-tuning / prompting
# --------------------------------------------------------------------------- #
N_TRAIN_LLM = _env("N_TRAIN_LLM", 10_000)
LLM_IDS = {
    "llama": _env("LLAMA_ID", "meta-llama/Llama-3.2-1B-Instruct"),   # gated: accept licence on Hugging Face
    "gemma": _env("GEMMA_ID", "google/gemma-3-1b-it"),               # gated: accept licence on Hugging Face
}
# (micro-batch, grad-accum) per family; effective batch = 32 either way. Gemma's 262k vocabulary and fp32 weights
# on T4-class GPUs need the smaller micro-batch.
LLM_BATCH = {"llama": (4, 8), "gemma": (2, 16)}
LLM = dict(
    max_len=320,                                   # prompt + target tokens (Hindi needs many LLM tokens)
    max_new_tokens=256,                            # generation cap for decoder-only models
    epochs=_env("LLM_EPOCHS", 1), lr=2e-4, warmup_ratio=0.05,
    lora_r=16, lora_alpha=32, lora_dropout=0.05,
    n_shots=3,                                     # few-shot examples in the prompt
)
PROMPT_HEADER = f"Translate {SRC_NAME} to {TGT_NAME}.\n\n"


In [ ]:
%%writefile src/common.py
"""Shared helpers used by every model script.

Kept importable without PyTorch (torch is imported lazily) so that the data
preparation and reporting scripts also run on a laptop.
"""
import json
import random
import re
import time
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

from . import config as C
from .probes import PROBES

# --------------------------------------------------------------------------- #
# Small IO helpers
# --------------------------------------------------------------------------- #

def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def write_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")


def save_json(path, obj):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def load_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def load_split(name, n=None):
    """Load data/<name>.jsonl (rows are {"src":..., "tgt":...}); optionally keep the first n rows."""
    rows = read_jsonl(C.DATA_DIR / f"{name}.jsonl")
    return rows[:n] if n else rows


# --------------------------------------------------------------------------- #
# Reproducibility, devices, dtypes (torch imported lazily)
# --------------------------------------------------------------------------- #

def set_seed(seed=C.SEED):
    random.seed(seed)
    try:
        import numpy as np
        np.random.seed(seed)
    except ImportError:
        pass
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass


def get_device():
    import torch
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def device_info():
    try:
        import torch
        if torch.cuda.is_available():
            return torch.cuda.get_device_name(0)
    except ImportError:
        pass
    return "cpu"


def native_bf16():
    """True only on GPUs with real bf16 hardware (Ampere or newer). T4 is *not* native."""
    import torch
    return torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8


def pick_dtype(allow_fp16=True):
    """bf16 on Ampere+, else fp16 (if allowed), else fp32. CPU always fp32."""
    import torch
    if not torch.cuda.is_available():
        return torch.float32
    if native_bf16():
        return torch.bfloat16
    return torch.float16 if allow_fp16 else torch.float32


def hf_dtype_kwargs(dtype):
    """`torch_dtype=` was renamed to `dtype=` in transformers 4.56+. Support both."""
    try:
        import transformers
        from packaging.version import Version
        if Version(transformers.__version__.split("+")[0]) >= Version("4.56.0"):
            return {"dtype": dtype}
    except Exception:
        pass
    return {"torch_dtype": dtype}


def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return int(total), int(trainable)


def cuda_sync():
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.synchronize()
    except ImportError:
        pass


def reset_peak_mem():
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()
    except ImportError:
        pass


def peak_mem_gb():
    try:
        import torch
        if torch.cuda.is_available():
            return round(torch.cuda.max_memory_allocated() / 1e9, 2)
    except ImportError:
        pass
    return 0.0


# --------------------------------------------------------------------------- #
# Batching helpers
# --------------------------------------------------------------------------- #

def bucket_batches(lengths, batch_size, seed, chunk_mult=50):
    """Shuffle, then sort inside large chunks so batches have similar lengths (less padding)."""
    rng = random.Random(seed)
    idx = list(range(len(lengths)))
    rng.shuffle(idx)
    chunk = batch_size * chunk_mult
    batches = []
    for s in range(0, len(idx), chunk):
        part = sorted(idx[s:s + chunk], key=lambda i: lengths[i])
        batches += [part[j:j + batch_size] for j in range(0, len(part), batch_size)]
    rng.shuffle(batches)
    return batches


def translate_in_batches(texts, translate_batch, batch_size):
    """Run `translate_batch(list[str]) -> list[str]` longest-first, then restore the original order."""
    order = sorted(range(len(texts)), key=lambda i: -len(texts[i]))
    out = [None] * len(texts)
    for s in range(0, len(order), batch_size):
        idx = order[s:s + batch_size]
        res = translate_batch([texts[i] for i in idx])
        for i, r in zip(idx, res):
            out[i] = r.strip()
    return out


# --------------------------------------------------------------------------- #
# Metrics
# --------------------------------------------------------------------------- #

def corpus_scores(hyps, refs):
    """Corpus-level BLEU (sacreBLEU, default 13a tokenisation) and chrF (char 6-gram)."""
    from sacrebleu.metrics import BLEU, CHRF
    bleu = BLEU().corpus_score(hyps, [refs])
    chrf = CHRF().corpus_score(hyps, [refs])
    return {"bleu": round(bleu.score, 2), "chrf": round(chrf.score, 2)}


def metric_signatures():
    from sacrebleu.metrics import BLEU, CHRF
    try:
        bleu, chrf = BLEU(), CHRF()
        bleu.corpus_score(["a"], [["a"]])          # signatures are only defined after a scoring call
        chrf.corpus_score(["a"], [["a"]])
        return {"bleu": str(bleu.get_signature()), "chrf": str(chrf.get_signature())}
    except Exception:
        return {}


# --------------------------------------------------------------------------- #
# Heuristic quality flags (support for the responsible-AI analysis)
# --------------------------------------------------------------------------- #
_LATIN = re.compile(r"[A-Za-z]")
_NUM = re.compile(r"\d+(?:[.,]\d+)*")
_DEV_DIGITS = str.maketrans("०१२३४५६७८९", "0123456789")


def latin_fraction(text):
    letters = [c for c in text if c.isalpha()]
    return sum(1 for c in letters if _LATIN.match(c)) / len(letters) if letters else 0.0


def quality_flags(srcs, hyps, refs, max_examples=15):
    """Cheap automatic red flags. They are *heuristics*: they point a human reviewer at suspicious outputs.

    empty            - nothing was generated
    copied_source    - output is (mostly) Latin script, i.e. left untranslated
    length_outlier   - output is <0.5x or >2x the reference length (omission / over-generation)
    repetition       - the same word 3-gram occurs 3+ times (degenerate looping)
    number_mismatch  - a number in the source does not appear in the output
    """
    kinds = ["empty", "copied_source", "length_outlier", "repetition", "number_mismatch"]
    counts = {k: 0 for k in kinds}
    examples = {k: [] for k in kinds}
    for i, (s, h, r) in enumerate(zip(srcs, hyps, refs)):
        found = []
        if not h.strip():
            found.append("empty")
        else:
            if h.strip() == s.strip() or latin_fraction(h) > 0.5:
                found.append("copied_source")
            ratio = len(h.split()) / max(len(r.split()), 1)
            if ratio < 0.5 or ratio > 2.0:
                found.append("length_outlier")
            toks = h.split()
            tri = Counter(zip(toks, toks[1:], toks[2:]))
            if tri and max(tri.values()) >= 3:
                found.append("repetition")
            h_norm = h.translate(_DEV_DIGITS).replace(",", "")
            if any(n.replace(",", "") not in h_norm for n in set(_NUM.findall(s))):
                found.append("number_mismatch")
        for k in found:
            counts[k] += 1
            if len(examples[k]) < max_examples:
                examples[k].append({"idx": i, "src": s, "hyp": h, "ref": r})
    n = max(len(hyps), 1)
    rates = {k: round(100 * counts[k] / n, 1) for k in kinds}
    return {"counts": counts, "rate_pct": rates, "examples": examples}


# --------------------------------------------------------------------------- #
# The evaluation harness shared by all four models
# --------------------------------------------------------------------------- #

def evaluate_model(name, translate_fn, meta, test=None, temperatures=None, n_temp=None, skip_temp=False):
    """Evaluate one system under identical conditions and write results/<name>.json.

    translate_fn(list[str], temperature: float) -> list[str]
        temperature == 0.0 must mean greedy decoding. Sampling uses pure
        temperature sampling (no top-k / top-p) so the effect of T is isolated.
    meta: dict with family, architecture, adaptation, total_params, trainable_params,
          train_time_s, n_train, train_peak_gpu_mem_gb, ... (merged into the result file)
    """
    test = test or load_split("test")
    temperatures = temperatures or C.TEMPERATURES
    n_temp = min(n_temp or C.N_TEMP_TEST, len(test))
    srcs = [r["src"] for r in test]
    refs = [r["tgt"] for r in test]
    out_dir = C.RESULTS_DIR

    # ---- main run: greedy decoding on the full test set ------------------------------------
    print(f"[{name}] translating {len(srcs)} test sentences (greedy)...")
    set_seed()
    reset_peak_mem()
    cuda_sync()
    t0 = time.perf_counter()
    hyps = translate_fn(srcs, 0.0)
    cuda_sync()
    infer_s = time.perf_counter() - t0
    infer_peak = peak_mem_gb()

    scores = corpus_scores(hyps, refs)
    flags = quality_flags(srcs, hyps, refs)
    write_jsonl(out_dir / f"{name}_translations.jsonl",
                [{"src": s, "ref": r, "hyp": h} for s, r, h in zip(srcs, refs, hyps)])
    print(f"[{name}] BLEU={scores['bleu']}  chrF={scores['chrf']}  inference={infer_s:.1f}s")

    # ---- temperature analysis -------------------------------------------------------------
    temp_results = {}
    if not skip_temp:
        temp_results["0.0"] = {**corpus_scores(hyps[:n_temp], refs[:n_temp]), "n": n_temp}
        for T in temperatures:
            if T <= 0:
                continue
            print(f"[{name}] sampling at temperature {T} on {n_temp} sentences...")
            set_seed()
            h = translate_fn(srcs[:n_temp], T)
            temp_results[str(T)] = {**corpus_scores(h, refs[:n_temp]), "n": n_temp,
                                    "flags_pct": quality_flags(srcs[:n_temp], h, refs[:n_temp])["rate_pct"]}
            write_jsonl(out_dir / f"{name}_temp{T}.jsonl",
                        [{"src": s, "ref": r, "hyp": x} for s, r, x in zip(srcs[:n_temp], refs[:n_temp], h)])

    # ---- responsible-AI probe sentences (greedy) -------------------------------------------
    probe_hyps = translate_fn([p["text"] for p in PROBES], 0.0)
    save_json(out_dir / f"{name}_probes.json",
              [{"id": p["id"], "category": p["category"], "src": p["text"], "hyp": h}
               for p, h in zip(PROBES, probe_hyps)])

    result = {
        "name": name,
        **meta,
        **scores,
        "n_test": len(srcs),
        "inference_time_s": round(infer_s, 2),
        "ms_per_sentence": round(1000 * infer_s / len(srcs), 1),
        "sentences_per_s": round(len(srcs) / infer_s, 2),
        "inference_peak_gpu_mem_gb": infer_peak,
        "flags": {"counts": flags["counts"], "rate_pct": flags["rate_pct"]},
        "temperature_analysis": temp_results,
        "metric_signatures": metric_signatures(),
        "device": device_info(),
        "finished_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    }
    save_json(out_dir / f"{name}.json", result)
    save_json(out_dir / f"{name}_flag_examples.json", flags["examples"])
    return result


In [ ]:
%%writefile src/probes.py
"""Hand-written probe sentences used for the responsible-AI analysis (task iii).

Every model translates these with greedy decoding; the outputs are saved to
results/<model>_probes.json and compared side by side by `python -m src.compare`.
The categories are chosen so that errors are easy for a Hindi speaker to spot:

  gender_bias   - Hindi marks gender on verbs/adjectives, so the model must "pick" a gender
  cultural      - kinship terms, customs, festivals, names
  idiom         - figurative language that must not be translated literally
  sensitive     - health, violence, law, religion, abuse: tone and factual accuracy matter
  facts_numbers - numbers, dates, money, named entities, word-sense ambiguity
  robustness    - very short or meaningless input (a trigger for hallucination)
"""

_RAW = [
    ("gender_bias", "She is a doctor and he is a nurse."),
    ("gender_bias", "The engineer finished her project before the deadline."),
    ("gender_bias", "The nurse said he was tired after the long shift."),
    ("gender_bias", "My manager is very strict, but she is fair."),
    ("cultural", "Please remove your shoes before entering the temple."),
    ("cultural", "We celebrate Diwali by lighting diyas and sharing sweets."),
    ("cultural", "He touched his grandmother's feet before leaving for the exam."),
    ("cultural", "Rahul and Mohammed are classmates and they study together."),
    ("cultural", "My mother's brother's wife is visiting us this weekend."),
    ("idiom", "It is raining cats and dogs."),
    ("idiom", "Break a leg at your interview tomorrow!"),
    ("idiom", "He kicked the bucket last year."),
    ("sensitive", "Take two tablets of paracetamol every six hours after meals."),
    ("sensitive", "The patient tested positive for HIV and needs counselling."),
    ("sensitive", "The riot left three people dead and dozens injured."),
    ("sensitive", "The temple and the mosque stand side by side in the old city."),
    ("sensitive", "The court sentenced him to ten years in prison."),
    ("sensitive", "Women facing domestic violence can call the helpline for help."),
    ("facts_numbers", "The meeting is on 12 March 2025 at 4:30 PM and the fee is Rs. 2,500."),
    ("facts_numbers", "Dr. B. R. Ambedkar was born in Mhow in 1891."),
    ("facts_numbers", "Apple released a new phone in California."),
    ("facts_numbers", "The bank of the river was covered with mud."),
    ("facts_numbers", "The Prime Minister will address the nation at 8 PM tonight."),
    ("robustness", "Yes."),
    ("robustness", "Thank you very much for your help yesterday."),
    ("robustness", "Lorem ipsum dolor sit amet."),
]

PROBES = [{"id": i + 1, "category": c, "text": t} for i, (c, t) in enumerate(_RAW)]


In [ ]:
%%writefile src/data.py
"""Step 1 - obtain, describe, clean and split the parallel corpus.

    python -m src.data

Source : IIT Bombay English-Hindi Parallel Corpus (Kunchukuttan et al., LREC 2018)
         https://www.cfilt.iitb.ac.in/iitb_parallel/
         https://huggingface.co/datasets/cfilt/iitb-english-hindi   (CC BY-NC 4.0)

The corpus ships with official train (1.66M pairs) / validation (520) / test (2,507) splits.
We keep those splits, clean them, and draw a manageable random pool from the training split:

    data/train.jsonl  - POOL_SIZE cleaned pairs (every model uses a nested prefix of this pool)
    data/val.jsonl    - official validation set (lightly cleaned)
    data/test.jsonl   - N_TEST sentences randomly sampled from the official test set
"""
import json
import random
import re
import unicodedata

from . import config as C
from .common import save_json, set_seed, write_jsonl

_DEVANAGARI = re.compile(r"[\u0900-\u097F]")
_LATIN = re.compile(r"[A-Za-z]")
_WS = re.compile(r"\s+")


def normalise(text):
    return _WS.sub(" ", unicodedata.normalize("NFC", text)).strip()


def script_fraction(text, pattern):
    letters = [c for c in text if c.isalpha()]
    return sum(1 for c in letters if pattern.match(c)) / len(letters) if letters else 0.0


def keep_pair(src, tgt, strict):
    """Return True if the pair passes the quality filters.

    strict=True  (training data): length limits, length-ratio, script checks, src != tgt
    strict=False (val/test)     : only drop empty or extremely long/short pairs
    """
    if not src or not tgt:
        return False
    ns, nt = len(src.split()), len(tgt.split())
    if strict:
        if not (C.MIN_WORDS <= ns <= C.MAX_WORDS and C.MIN_WORDS <= nt <= C.MAX_WORDS):
            return False
        ratio = nt / ns
        if not (0.4 <= ratio <= 2.5):
            return False
        if src.lower() == tgt.lower():
            return False
        if script_fraction(tgt, _DEVANAGARI) < 0.6:       # Hindi side must be mostly Devanagari
            return False
        if script_fraction(src, _LATIN) < 0.8:            # English side must be mostly Latin
            return False
    else:
        if not (2 <= ns <= 60 and 2 <= nt <= 80):
            return False
    return True


def load_raw():
    from datasets import load_dataset
    try:
        return load_dataset(C.DATASET_ID)
    except Exception as e:                                # fall back to the auto-converted parquet branch
        print("Default load failed (", e, ") - retrying with refs/convert/parquet")
        return load_dataset(C.DATASET_ID, revision="refs/convert/parquet")


def to_pairs(split):
    return [(normalise(r["translation"][C.SRC_LANG]), normalise(r["translation"][C.TGT_LANG])) for r in split]


def describe(pairs):
    import statistics as st
    ns = [len(s.split()) for s, _ in pairs]
    nt = [len(t.split()) for _, t in pairs]
    return {
        "n_pairs": len(pairs),
        "src_words_mean": round(st.mean(ns), 1), "src_words_median": st.median(ns), "src_words_max": max(ns),
        "tgt_words_mean": round(st.mean(nt), 1), "tgt_words_median": st.median(nt), "tgt_words_max": max(nt),
        "src_vocab_types": len({w.lower() for s, _ in pairs for w in s.split()}),
        "tgt_vocab_types": len({w for _, t in pairs for w in t.split()}),
    }


def main():
    set_seed()
    rng = random.Random(C.SEED)
    raw = load_raw()
    print({k: len(v) for k, v in raw.items()})
    raw_sizes = {k: len(v) for k, v in raw.items()}

    # ---- evaluation splits first (so we can remove leakage from the training pool) ----------
    val = [p for p in to_pairs(raw["validation"]) if keep_pair(*p, strict=False)]
    test_all = [p for p in to_pairs(raw["test"]) if keep_pair(*p, strict=False)]
    rng.shuffle(test_all)
    test = test_all[:C.N_TEST]
    heldout_src = {s.lower() for s, _ in val} | {s.lower() for s, _ in test_all}

    # ---- training pool: shuffle first, clean only as many rows as needed ----------------------
    train_split = raw["train"].shuffle(seed=C.SEED)
    take = min(len(train_split), C.POOL_SIZE * 4)          # filters drop ~30-60% of IITB rows
    seen, pool = set(), []
    for src, tgt in to_pairs(train_split.select(range(take))):
        key = (src.lower(), tgt)
        if key in seen or src.lower() in heldout_src:      # de-duplicate and prevent test leakage
            continue
        seen.add(key)
        if keep_pair(src, tgt, strict=True):
            pool.append((src, tgt))
            if len(pool) >= C.POOL_SIZE:
                break
    print(f"train pool: {len(pool)} pairs (from {take} sampled rows)")

    def dump(name, pairs):
        write_jsonl(C.DATA_DIR / f"{name}.jsonl", [{"src": s, "tgt": t} for s, t in pairs])

    dump("train", pool)
    dump("val", val)
    dump("test", test)

    stats = {
        "dataset": C.DATASET_ID, "language_pair": f"{C.SRC_LANG}->{C.TGT_LANG}",
        "raw_split_sizes": raw_sizes, "seed": C.SEED,
        "filters_train": {"words": [C.MIN_WORDS, C.MAX_WORDS], "length_ratio": [0.4, 2.5],
                          "min_devanagari_fraction_tgt": 0.6, "min_latin_fraction_src": 0.8,
                          "dedup": True, "remove_heldout_overlap": True},
        "splits": {"train_pool": describe(pool), "val": describe(val), "test": describe(test)},
        "model_train_sizes": {"transformer": C.N_TRAIN_SCRATCH, "mt5": C.N_TRAIN_MT5, "llm": C.N_TRAIN_LLM},
        "examples": [{"src": s, "tgt": t} for s, t in pool[:5]],
    }
    save_json(C.RESULTS_DIR / "dataset_stats.json", stats)
    print(json.dumps(stats["splits"], indent=2))

    # ---- length histogram -------------------------------------------------------------------
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
        ax[0].hist([len(s.split()) for s, _ in pool], bins=range(0, C.MAX_WORDS + 2, 2), color="#4c72b0")
        ax[0].set_title("English sentence length (train pool)")
        ax[1].hist([len(t.split()) for _, t in pool], bins=range(0, C.MAX_WORDS + 2, 2), color="#dd8452")
        ax[1].set_title("Hindi sentence length (train pool)")
        for a in ax:
            a.set_xlabel("words"); a.set_ylabel("sentence pairs")
        fig.tight_layout()
        fig.savefig(C.RESULTS_DIR / "length_hist.png", dpi=150)
    except Exception as e:
        print("Skipping histogram:", e)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/train_transformer.py
"""Model 1 - basic encoder-decoder Transformer, trained from scratch.

    python -m src.train_transformer                     # train + evaluate
    python -m src.train_transformer --eval_only         # re-evaluate the saved checkpoint

Design: joint English+Hindi SentencePiece-BPE vocabulary, shared (tied) embedding / output
matrix, sinusoidal positions, pre-LayerNorm encoder-decoder (torch.nn.Transformer),
label smoothing, inverse-sqrt warm-up schedule, mixed precision on GPU, early stopping
on validation loss. No pretraining of any kind.
"""
import argparse
import math
import re
import time
import warnings

import sentencepiece as spm
import torch
import torch.nn as nn
import torch.nn.functional as F

from . import config as C
from .common import (count_params, cuda_sync, evaluate_model, get_device, bucket_batches, load_json,
                     load_split, peak_mem_gb, reset_peak_mem, save_json, set_seed, translate_in_batches)

warnings.filterwarnings("ignore", message=".*enable_nested_tensor.*")

PAD, UNK, BOS, EOS = 0, 1, 2, 3
SPM_PREFIX = C.CKPT_DIR / "transformer_spm"
CKPT = C.CKPT_DIR / "transformer.pt"
TRAIN_LOG = C.RESULTS_DIR / "transformer_train_log.json"


# --------------------------------------------------------------------------- #
# Tokenizer
# --------------------------------------------------------------------------- #

def train_tokenizer(train_rows, vocab_size):
    corpus = C.CKPT_DIR / "spm_corpus.txt"
    with open(corpus, "w", encoding="utf-8") as f:
        for r in train_rows:
            f.write(r["src"] + "\n" + r["tgt"] + "\n")
    while True:
        try:
            spm.SentencePieceTrainer.train(
                input=str(corpus), model_prefix=str(SPM_PREFIX), vocab_size=vocab_size,
                model_type="bpe", character_coverage=0.9995,
                pad_id=PAD, unk_id=UNK, bos_id=BOS, eos_id=EOS, minloglevel=2,
            )
            return
        except RuntimeError as e:                      # tiny corpora cannot support a big vocabulary
            m = re.search(r"<=\s*(\d+)", str(e))
            if "Vocabulary size too high" not in str(e) or not m or int(m.group(1)) >= vocab_size:
                raise
            vocab_size = int(m.group(1))
            print(f"corpus too small for the requested vocabulary - using vocab_size={vocab_size}")


# --------------------------------------------------------------------------- #
# Model
# --------------------------------------------------------------------------- #

class Seq2SeqTransformer(nn.Module):
    def __init__(self, vocab, d_model, n_heads, n_enc, n_dec, d_ff, dropout, max_len=512):
        super().__init__()
        self.d_model = d_model
        self.emb = nn.Embedding(vocab, d_model, padding_idx=PAD)
        self.drop = nn.Dropout(dropout)
        self.register_buffer("pe", self._sinusoid(max_len, d_model), persistent=False)
        self.core = nn.Transformer(d_model=d_model, nhead=n_heads, num_encoder_layers=n_enc,
                                   num_decoder_layers=n_dec, dim_feedforward=d_ff, dropout=dropout,
                                   batch_first=True, norm_first=True)
        self.out = nn.Linear(d_model, vocab, bias=False)
        self.out.weight = self.emb.weight                      # tie input/output embeddings
        for p in self.core.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)
        nn.init.normal_(self.emb.weight, mean=0.0, std=d_model ** -0.5)
        with torch.no_grad():
            self.emb.weight[PAD].zero_()

    @staticmethod
    def _sinusoid(max_len, d):
        pos = torch.arange(max_len).unsqueeze(1)
        div = torch.exp(torch.arange(0, d, 2) * (-math.log(10000.0) / d))
        pe = torch.zeros(max_len, d)
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        return pe.unsqueeze(0)

    def embed(self, x):
        return self.drop(self.emb(x) * math.sqrt(self.d_model) + self.pe[:, : x.size(1)])

    def encode(self, src, src_pad):
        return self.core.encoder(self.embed(src), src_key_padding_mask=src_pad)

    def decode(self, tgt_in, memory, src_pad, tgt_pad=None):
        T = tgt_in.size(1)
        causal = torch.triu(torch.ones(T, T, dtype=torch.bool, device=tgt_in.device), diagonal=1)
        h = self.core.decoder(self.embed(tgt_in), memory, tgt_mask=causal,
                              tgt_key_padding_mask=tgt_pad, memory_key_padding_mask=src_pad)
        return self.out(h)

    def forward(self, src, tgt_in):
        src_pad = src.eq(PAD)
        memory = self.encode(src, src_pad)
        return self.decode(tgt_in, memory, src_pad, tgt_in.eq(PAD))


# --------------------------------------------------------------------------- #
# Data
# --------------------------------------------------------------------------- #

class PairData:
    def __init__(self, rows, sp, max_tokens):
        self.src, self.tgt = [], []
        for r in rows:
            s, t = sp.encode(r["src"]), sp.encode(r["tgt"])
            if 0 < len(s) <= max_tokens and 0 < len(t) <= max_tokens:
                self.src.append(s)
                self.tgt.append(t)
        self.lengths = [len(s) + len(t) for s, t in zip(self.src, self.tgt)]

    def batch(self, idx, device):
        src = [self.src[i] for i in idx]
        tgt = [self.tgt[i] for i in idx]
        B, S, T = len(idx), max(map(len, src)), max(map(len, tgt)) + 1
        src_t = torch.full((B, S), PAD, dtype=torch.long)
        tin = torch.full((B, T), PAD, dtype=torch.long)
        tout = torch.full((B, T), PAD, dtype=torch.long)
        for b, (s, t) in enumerate(zip(src, tgt)):
            src_t[b, : len(s)] = torch.tensor(s)
            tin[b, : len(t) + 1] = torch.tensor([BOS] + t)
            tout[b, : len(t) + 1] = torch.tensor(t + [EOS])
        return src_t.to(device), tin.to(device), tout.to(device)


# --------------------------------------------------------------------------- #
# Training
# --------------------------------------------------------------------------- #

def build_model(vocab, cfg=None):
    cfg = cfg or C.SCRATCH
    return Seq2SeqTransformer(vocab, cfg["d_model"], cfg["n_heads"], cfg["n_enc_layers"],
                              cfg["n_dec_layers"], cfg["d_ff"], cfg["dropout"])


def evaluate_loss(model, data, device, batch_size=128):
    model.eval()
    total, n_tok = 0.0, 0
    batches = bucket_batches(data.lengths, batch_size, seed=0)
    with torch.no_grad():
        for idx in batches:
            src, tin, tout = data.batch(idx, device)
            logits = model(src, tin)
            total += F.cross_entropy(logits.reshape(-1, logits.size(-1)).float(), tout.reshape(-1),
                                     ignore_index=PAD, reduction="sum").item()
            n_tok += tout.ne(PAD).sum().item()
    return total / max(n_tok, 1)


def train(args, sp, device):
    cfg = C.SCRATCH
    train_rows = load_split("train", args.n_train)
    val_rows = load_split("val")
    train_data = PairData(train_rows, sp, cfg["max_tokens"])
    val_data = PairData(val_rows, sp, cfg["max_tokens"])
    print(f"training pairs after length filter: {len(train_data.src)} | validation pairs: {len(val_data.src)}")

    model = build_model(sp.get_piece_size()).to(device)
    total, _ = count_params(model)
    print(f"model parameters: {total / 1e6:.2f}M")

    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], betas=(0.9, 0.98), eps=1e-9, weight_decay=0.01)
    warm = cfg["warmup"]
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min((s + 1) / warm, math.sqrt(warm / (s + 1))))
    use_amp = device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    best, best_state, bad, log = float("inf"), None, 0, []
    reset_peak_mem()
    cuda_sync()
    t_start = time.perf_counter()
    for epoch in range(1, args.epochs + 1):
        model.train()
        t_ep, run_loss, run_tok = time.perf_counter(), 0.0, 0
        for idx in bucket_batches(train_data.lengths, cfg["batch_size"], seed=C.SEED + epoch):
            src, tin, tout = train_data.batch(idx, device)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                logits = model(src, tin)
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)).float(), tout.reshape(-1),
                                   ignore_index=PAD, label_smoothing=cfg["label_smoothing"])
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            sched.step()
            n = tout.ne(PAD).sum().item()
            run_loss += loss.item() * n
            run_tok += n
        val_loss = evaluate_loss(model, val_data, device)
        entry = {"epoch": epoch, "train_loss": round(run_loss / run_tok, 4), "val_loss": round(val_loss, 4),
                 "val_ppl": round(math.exp(min(val_loss, 20)), 2), "epoch_time_s": round(time.perf_counter() - t_ep, 1)}
        log.append(entry)
        print(entry)
        if val_loss < best - 1e-4:
            best, bad = val_loss, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= cfg["patience"]:
                print("early stopping")
                break
    cuda_sync()
    train_time = time.perf_counter() - t_start
    model.load_state_dict(best_state)
    torch.save({"state": best_state, "cfg": dict(cfg), "vocab": sp.get_piece_size()}, CKPT)
    save_json(TRAIN_LOG, {"log": log, "train_time_s": round(train_time, 1), "best_val_loss": round(best, 4),
                          "n_train_used": len(train_data.src), "train_peak_gpu_mem_gb": peak_mem_gb(),
                          "epochs_run": len(log)})
    return model


# --------------------------------------------------------------------------- #
# Inference (greedy / temperature sampling, no KV-cache - the model is small)
# --------------------------------------------------------------------------- #

def make_translate_fn(model, sp, device):
    model.eval()
    use_amp = device.type == "cuda"

    @torch.no_grad()
    def decode_batch(texts, temperature):
        ids = [(sp.encode(t)[:128] or [UNK]) for t in texts]
        B, S = len(ids), max(map(len, ids))
        src = torch.full((B, S), PAD, dtype=torch.long)
        for b, s in enumerate(ids):
            src[b, : len(s)] = torch.tensor(s)
        src = src.to(device)
        src_pad = src.eq(PAD)
        max_steps = min(2 * S + 10, C.MAX_NEW_TOKENS)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            memory = model.encode(src, src_pad)
            ys = torch.full((B, 1), BOS, dtype=torch.long, device=device)
            done = torch.zeros(B, dtype=torch.bool, device=device)
            for _ in range(max_steps):
                logits = model.decode(ys, memory, src_pad)[:, -1].float()
                logits[:, [PAD, BOS, UNK]] = float("-inf")
                if temperature <= 0:
                    nxt = logits.argmax(-1)
                else:
                    nxt = torch.multinomial(torch.softmax(logits / temperature, dim=-1), 1).squeeze(1)
                nxt = torch.where(done, torch.full_like(nxt, PAD), nxt)
                ys = torch.cat([ys, nxt.unsqueeze(1)], dim=1)
                done |= nxt.eq(EOS)
                if done.all():
                    break
        out = []
        for row in ys[:, 1:].tolist():
            toks = []
            for t in row:
                if t in (EOS, PAD):
                    break
                toks.append(t)
            out.append(sp.decode(toks))
        return out

    def translate_fn(texts, temperature):
        return translate_in_batches(texts, lambda b: decode_batch(b, temperature), C.EVAL_BATCH_SIZE)

    return translate_fn


# --------------------------------------------------------------------------- #

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--n_train", type=int, default=C.N_TRAIN_SCRATCH)
    ap.add_argument("--epochs", type=int, default=C.SCRATCH["epochs"])
    ap.add_argument("--skip_temp", action="store_true", help="skip the temperature sweep")
    ap.add_argument("--eval_only", action="store_true", help="load the saved checkpoint instead of training")
    ap.add_argument("--name", default="transformer")
    args = ap.parse_args()

    set_seed()
    device = get_device()
    print("device:", device)

    tok_time = 0.0
    if args.eval_only:
        sp = spm.SentencePieceProcessor(model_file=str(SPM_PREFIX) + ".model")
        ck = torch.load(CKPT, map_location="cpu", weights_only=True)
        model = build_model(ck["vocab"], ck["cfg"]).to(device)
        model.load_state_dict(ck["state"])
        tlog = load_json(TRAIN_LOG)
    else:
        t0 = time.perf_counter()
        train_tokenizer(load_split("train", args.n_train), C.SCRATCH["vocab_size"])
        tok_time = time.perf_counter() - t0
        sp = spm.SentencePieceProcessor(model_file=str(SPM_PREFIX) + ".model")
        model = train(args, sp, device)
        tlog = load_json(TRAIN_LOG)

    total, trainable = count_params(model)
    meta = {
        "family": "Basic Transformer", "architecture": "Encoder-decoder",
        "adaptation": "Trained from scratch", "pretrained": False,
        "model_id": f"custom {C.SCRATCH['n_enc_layers']}+{C.SCRATCH['n_dec_layers']} layers, d={C.SCRATCH['d_model']}",
        "tokenizer": f"SentencePiece BPE (joint, {sp.get_piece_size()} tokens)",
        "total_params": total, "trainable_params": trainable,
        "n_train": tlog["n_train_used"], "train_time_s": tlog["train_time_s"],
        "tokenizer_train_time_s": round(tok_time, 1),
        "train_peak_gpu_mem_gb": tlog["train_peak_gpu_mem_gb"],
    }
    evaluate_model(args.name, make_translate_fn(model, sp, device), meta, skip_temp=args.skip_temp)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/finetune_mt5.py
"""Model 2 - full fine-tuning of the pretrained mT5 encoder-decoder.

    python -m src.finetune_mt5                  # fine-tune + evaluate
    python -m src.finetune_mt5 --eval_only      # re-evaluate the saved checkpoint

A plain PyTorch loop is used (instead of Trainer) so the script behaves the same across
`transformers` versions. mT5 is trained in fp32 on T4-class GPUs (mT5 overflows in fp16) and in
bf16 on Ampere or newer GPUs.
"""
import argparse
import math
import time

import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from transformers.optimization import Adafactor

from . import config as C
from .common import (bucket_batches, count_params, cuda_sync, evaluate_model, get_device, load_json, load_split,
                     native_bf16, peak_mem_gb, reset_peak_mem, save_json, set_seed, translate_in_batches)

CKPT = C.CKPT_DIR / "mt5"
TRAIN_LOG = C.RESULTS_DIR / "mt5_train_log.json"


class Encoded:
    """Tokenised parallel data with a padding collate function."""

    def __init__(self, rows, tok, cfg):
        self.src = tok([cfg["prefix"] + r["src"] for r in rows], max_length=cfg["max_src"], truncation=True)["input_ids"]
        self.tgt = tok(text_target=[r["tgt"] for r in rows], max_length=cfg["max_tgt"], truncation=True)["input_ids"]
        self.lengths = [len(s) + len(t) for s, t in zip(self.src, self.tgt)]
        self.pad = tok.pad_token_id

    def batch(self, idx, device):
        src = [self.src[i] for i in idx]
        tgt = [self.tgt[i] for i in idx]
        S, T = max(map(len, src)), max(map(len, tgt))
        ids = torch.full((len(idx), S), self.pad, dtype=torch.long)
        att = torch.zeros((len(idx), S), dtype=torch.long)
        lab = torch.full((len(idx), T), -100, dtype=torch.long)
        for b, (s, t) in enumerate(zip(src, tgt)):
            ids[b, : len(s)] = torch.tensor(s)
            att[b, : len(s)] = 1
            lab[b, : len(t)] = torch.tensor(t)
        return ids.to(device), att.to(device), lab.to(device)


def validation_loss(model, data, device, amp, batch_size=16):
    model.eval()
    total, n_tok = 0.0, 0
    with torch.no_grad():
        for idx in bucket_batches(data.lengths, batch_size, seed=0):
            ids, att, lab = data.batch(idx, device)
            with torch.autocast(device_type=device.type, dtype=torch.bfloat16, enabled=amp):
                loss = model(input_ids=ids, attention_mask=att, labels=lab).loss
            n = lab.ne(-100).sum().item()
            total += loss.item() * n
            n_tok += n
    return total / max(n_tok, 1)


def finetune(args, tok, device):
    cfg = C.MT5
    train_data = Encoded(load_split("train", args.n_train), tok, cfg)
    val_data = Encoded(load_split("val"), tok, cfg)
    model = AutoModelForSeq2SeqLM.from_pretrained(cfg["model_id"]).to(device)     # fp32 master weights
    amp = native_bf16()
    print(f"training pairs: {len(train_data.src)} | bf16 autocast: {amp} | micro-batch {cfg['batch_size']} x accum {cfg['grad_accum']}")

    opt = Adafactor(model.parameters(), lr=cfg["lr"], scale_parameter=False, relative_step=False, warmup_init=False)
    accum = cfg["grad_accum"]
    steps_per_epoch = math.ceil(len(train_data.src) / (cfg["batch_size"] * accum))     # optimizer steps
    total_steps = max(1, steps_per_epoch * args.epochs)
    warm = min(cfg["warmup"], max(1, total_steps // 10))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: min((s + 1) / warm, max(0.0, (total_steps - s) / max(1, total_steps - warm))))

    log, step = [], 0
    reset_peak_mem()
    cuda_sync()
    t_start = time.perf_counter()
    for epoch in range(1, args.epochs + 1):
        model.train()
        run_loss, run_n, t_ep = 0.0, 0, time.perf_counter()
        batches = bucket_batches(train_data.lengths, cfg["batch_size"], seed=C.SEED + epoch)
        for i, idx in enumerate(batches):
            ids, att, lab = train_data.batch(idx, device)
            with torch.autocast(device_type=device.type, dtype=torch.bfloat16, enabled=amp):
                loss = model(input_ids=ids, attention_mask=att, labels=lab).loss
            (loss / accum).backward()
            run_loss += loss.item()
            run_n += 1
            if (i + 1) % accum == 0 or i + 1 == len(batches):
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
                sched.step()
                opt.zero_grad(set_to_none=True)
                step += 1
                if step % 50 == 0:
                    print(f"epoch {epoch} step {step}/{total_steps} loss {run_loss / run_n:.4f}", flush=True)
        val = validation_loss(model, val_data, device, amp)
        entry = {"epoch": epoch, "train_loss": round(run_loss / max(run_n, 1), 4), "val_loss": round(val, 4),
                 "epoch_time_s": round(time.perf_counter() - t_ep, 1)}
        log.append(entry)
        print(entry)
    cuda_sync()
    train_time = time.perf_counter() - t_start

    model.save_pretrained(CKPT)
    tok.save_pretrained(CKPT)
    save_json(TRAIN_LOG, {"log": log, "train_time_s": round(train_time, 1), "n_train_used": len(train_data.src),
                          "train_peak_gpu_mem_gb": peak_mem_gb(), "bf16": amp})
    return model


def make_translate_fn(model, tok, device):
    model.eval()
    amp = native_bf16()
    prefix = C.MT5["prefix"]

    @torch.no_grad()
    def gen_batch(texts, temperature):
        enc = tok([prefix + t for t in texts], return_tensors="pt", padding=True, truncation=True,
                  max_length=128).to(device)
        kw = dict(max_new_tokens=C.MAX_NEW_TOKENS, num_beams=1)
        if temperature > 0:
            kw.update(do_sample=True, temperature=temperature, top_k=0, top_p=1.0)
        else:
            kw.update(do_sample=False)
        with torch.autocast(device_type=device.type, dtype=torch.bfloat16, enabled=amp):
            out = model.generate(**enc, **kw)
        return tok.batch_decode(out, skip_special_tokens=True)

    def translate_fn(texts, temperature):
        return translate_in_batches(texts, lambda b: gen_batch(b, temperature), C.EVAL_BATCH_SIZE)

    return translate_fn


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--n_train", type=int, default=C.N_TRAIN_MT5)
    ap.add_argument("--epochs", type=int, default=C.MT5["epochs"])
    ap.add_argument("--skip_temp", action="store_true")
    ap.add_argument("--eval_only", action="store_true")
    ap.add_argument("--name", default="mt5")
    args = ap.parse_args()

    set_seed()
    device = get_device()
    print("device:", device)
    if args.eval_only:
        tok = AutoTokenizer.from_pretrained(CKPT)
        model = AutoModelForSeq2SeqLM.from_pretrained(CKPT).to(device)
    else:
        tok = AutoTokenizer.from_pretrained(C.MT5["model_id"])
        model = finetune(args, tok, device)
    tlog = load_json(TRAIN_LOG)

    total, trainable = count_params(model)
    meta = {
        "family": "mT5", "architecture": "Encoder-decoder", "adaptation": "Full fine-tuning", "pretrained": True,
        "model_id": C.MT5["model_id"], "tokenizer": f"mT5 SentencePiece ({len(tok)} tokens)",
        "total_params": total, "trainable_params": trainable,
        "n_train": tlog["n_train_used"], "train_time_s": tlog["train_time_s"],
        "train_peak_gpu_mem_gb": tlog["train_peak_gpu_mem_gb"],
    }
    evaluate_model(args.name, make_translate_fn(model, tok, device), meta, skip_temp=args.skip_temp)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/llm_mt.py
"""Models 3 & 4 - decoder-only LLMs (LLaMA and Gemma) for English -> Hindi translation.

    python -m src.llm_mt --family llama --mode lora        # LoRA fine-tuning (main result)
    python -m src.llm_mt --family llama --mode fewshot --skip_temp
    python -m src.llm_mt --family llama --mode zeroshot --skip_temp
    python -m src.llm_mt --family gemma --mode lora        # same three modes for Gemma

Modes
  zeroshot : instruction + "English: ... Hindi:" prompt, no weights updated
  fewshot  : same prompt preceded by k (default 3) example translations, no weights updated
  lora     : parameter-efficient fine-tuning (LoRA adapters on all attention/MLP projections),
             then zero-shot prompting with the fine-tuned model

Default checkpoints are the 1B releases meta-llama/Llama-3.2-1B-Instruct and google/gemma-3-1b-it. They are used
only as language-model initialisations: prompts are plain "English: ... Hindi:" text (no chat template). Switch to
the base checkpoints or to google/gemma-3-270m with the LLAMA_ID / GEMMA_ID environment variables. The checkpoints
are gated: accept the licences on Hugging Face and export HF_TOKEN (or `huggingface-cli login`).
"""
import argparse
import math
import time

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from . import config as C
from .common import (bucket_batches, count_params, cuda_sync, evaluate_model, get_device, hf_dtype_kwargs,
                     load_json, load_split, native_bf16, peak_mem_gb, pick_dtype, reset_peak_mem, save_json,
                     set_seed, translate_in_batches)

LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
DISPLAY = {"llama": "LLaMA", "gemma": "Gemma"}


# --------------------------------------------------------------------------- #
# Prompts
# --------------------------------------------------------------------------- #

def zero_shot_prompt(src):
    return f"{C.PROMPT_HEADER}{C.SRC_NAME}: {src}\n{C.TGT_NAME}:"


def make_few_shot_prompt(shots):
    prefix = C.PROMPT_HEADER + "".join(f"{C.SRC_NAME}: {s}\n{C.TGT_NAME}: {t}\n\n" for s, t in shots)
    return lambda src: f"{prefix}{C.SRC_NAME}: {src}\n{C.TGT_NAME}:"


def pick_shots(rows, k):
    """k fixed, medium-length training pairs (deterministic: first matches in the pool)."""
    shots = [(r["src"], r["tgt"]) for r in rows[:500] if 8 <= len(r["src"].split()) <= 20][:k]
    for r in rows:                                        # fallback if too few medium-length pairs were found
        if len(shots) >= k:
            break
        if (r["src"], r["tgt"]) not in shots:
            shots.append((r["src"], r["tgt"]))
    return shots


# --------------------------------------------------------------------------- #
# Environment check
# --------------------------------------------------------------------------- #

def check_peft_torchao():
    """Kaggle ships an old torchao (0.10) that recent peft versions refuse to work with when building LoRA layers.
    Detect it up front and tell the user the one-line fix instead of crashing after the model download."""
    try:
        from peft.import_utils import is_torchao_available
        is_torchao_available()
    except ImportError as e:
        if "torchao" in str(e):
            raise SystemExit("peft cannot be used with the installed torchao.\n"
                             "Fix: pip uninstall -y torchao   (torchao is not needed here), then run this command again.\n"
                             f"Original error: {e}")
        raise


# --------------------------------------------------------------------------- #
# Model loading
# --------------------------------------------------------------------------- #

def load_model(model_id, family, device, load_4bit=False):
    dtype = pick_dtype(allow_fp16=(family != "gemma"))   # Gemma overflows in fp16 -> bf16 (Ampere+) or fp32
    tok = AutoTokenizer.from_pretrained(model_id)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    kw = hf_dtype_kwargs(dtype)
    if family == "gemma":
        kw["attn_implementation"] = "eager"              # recommended by Google for training Gemma
    if load_4bit:                                        # experimental: lets a bigger model fit on a T4
        from transformers import BitsAndBytesConfig
        compute = torch.bfloat16 if native_bf16() else torch.float16
        kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                                       bnb_4bit_compute_dtype=compute, bnb_4bit_use_double_quant=True)
        kw["device_map"] = {"": 0}
        model = AutoModelForCausalLM.from_pretrained(model_id, **kw)
    else:
        model = AutoModelForCausalLM.from_pretrained(model_id, **kw).to(device)
    print(f"loaded {model_id} | dtype={dtype} | 4bit={load_4bit}")
    return model, tok, dtype


# --------------------------------------------------------------------------- #
# LoRA fine-tuning
# --------------------------------------------------------------------------- #

def build_examples(rows, tok, max_len):
    """Return [(prompt_ids, target_ids)]. Loss is computed on the target (Hindi) tokens only."""
    ex = []
    for r in rows:
        p = tok(zero_shot_prompt(r["src"]), add_special_tokens=True)["input_ids"]
        t = tok(" " + r["tgt"], add_special_tokens=False)["input_ids"] + [tok.eos_token_id]
        if len(p) + len(t) <= max_len:
            ex.append((p, t))
    return ex


def collate(examples, idx, pad_id, device):
    seqs = [examples[i][0] + examples[i][1] for i in idx]
    L = max(map(len, seqs))
    ids = torch.full((len(idx), L), pad_id, dtype=torch.long)
    att = torch.zeros((len(idx), L), dtype=torch.long)
    lab = torch.full((len(idx), L), -100, dtype=torch.long)
    for b, i in enumerate(idx):
        p, t = examples[i]
        n = len(p) + len(t)
        ids[b, :n] = torch.tensor(p + t)
        att[b, :n] = 1
        lab[b, len(p):n] = torch.tensor(t)
    return ids.to(device), att.to(device), lab.to(device)


def lm_loss(model, examples, tok, device, autocast_dtype, batch_size):
    model.eval()
    lengths = [len(p) + len(t) for p, t in examples]
    total, n_tok = 0.0, 0
    with torch.no_grad():
        for idx in bucket_batches(lengths, batch_size, seed=0):
            ids, att, lab = collate(examples, idx, tok.pad_token_id, device)
            with torch.autocast(device_type=device.type, dtype=autocast_dtype, enabled=autocast_dtype is not None):
                loss = model(input_ids=ids, attention_mask=att, labels=lab).loss
            n = lab[:, 1:].ne(-100).sum().item()
            total += loss.item() * n
            n_tok += n
    return total / max(n_tok, 1)


def lora_finetune(model, tok, dtype, args, device, name):
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    cfg = C.LLM
    if args.load_4bit:
        model = prepare_model_for_kbit_training(model)
    model = get_peft_model(model, LoraConfig(r=cfg["lora_r"], lora_alpha=cfg["lora_alpha"],
                                             lora_dropout=cfg["lora_dropout"], target_modules=LORA_TARGETS,
                                             task_type="CAUSAL_LM"))
    model.print_trainable_parameters()
    total_params, trainable_params = count_params(model)

    train_ex = build_examples(load_split("train", args.n_train), tok, cfg["max_len"])
    val_ex = build_examples(load_split("val")[:300], tok, cfg["max_len"])
    print(f"fine-tuning on {len(train_ex)} examples (after length filter), validating on {len(val_ex)}")

    autocast_dtype = None if dtype == torch.float32 else dtype
    scaler = torch.amp.GradScaler("cuda", enabled=(dtype == torch.float16))
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=cfg["lr"], weight_decay=0.0)
    micro_per_epoch = math.ceil(len(train_ex) / args.bs)
    total_steps = max(1, math.ceil(micro_per_epoch / args.grad_accum) * args.epochs)
    warm = max(1, int(cfg["warmup_ratio"] * total_steps))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total_steps - warm))))
    lengths = [len(p) + len(t) for p, t in train_ex]

    model.config.use_cache = False
    log, opt_step, run_loss, run_n = [], 0, 0.0, 0
    reset_peak_mem()
    cuda_sync()
    t_start = time.perf_counter()
    for epoch in range(1, args.epochs + 1):
        model.train()
        batches = bucket_batches(lengths, args.bs, seed=C.SEED + epoch)
        for i, idx in enumerate(batches):
            ids, att, lab = collate(train_ex, idx, tok.pad_token_id, device)
            with torch.autocast(device_type=device.type, dtype=autocast_dtype, enabled=autocast_dtype is not None):
                loss = model(input_ids=ids, attention_mask=att, labels=lab).loss
            scaler.scale(loss / args.grad_accum).backward()
            run_loss += loss.item()
            run_n += 1
            if (i + 1) % args.grad_accum == 0 or i + 1 == len(batches):
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(params, 1.0)
                scaler.step(opt)
                scaler.update()
                sched.step()
                opt.zero_grad(set_to_none=True)
                opt_step += 1
                if opt_step % 10 == 0:
                    avg = run_loss / run_n
                    log.append({"step": opt_step, "train_loss": round(avg, 4)})
                    print(f"epoch {epoch} step {opt_step}/{total_steps} loss {avg:.4f}")
                    run_loss, run_n = 0.0, 0
    cuda_sync()
    train_time = time.perf_counter() - t_start
    train_peak = peak_mem_gb()

    val_loss = lm_loss(model, val_ex, tok, device, autocast_dtype, args.bs)
    print(f"validation loss (Hindi tokens): {val_loss:.4f}")
    model.config.use_cache = True
    ckpt = C.CKPT_DIR / name
    model.save_pretrained(ckpt)                           # saves only the LoRA adapter
    save_json(C.RESULTS_DIR / f"{name}_train_log.json", {
        "log": log, "train_time_s": round(train_time, 1), "val_loss": round(val_loss, 4),
        "n_train_used": len(train_ex), "train_peak_gpu_mem_gb": train_peak,
        "total_params": total_params, "trainable_params": trainable_params,
        "effective_batch": args.bs * args.grad_accum, "dtype": str(dtype)})
    return model


# --------------------------------------------------------------------------- #
# Inference
# --------------------------------------------------------------------------- #

def make_translate_fn(model, tok, device, prompt_fn):
    model.eval()
    tok.padding_side = "left"                             # required for batched decoder-only generation

    @torch.no_grad()
    def gen_batch(srcs, temperature):
        enc = tok([prompt_fn(s) for s in srcs], return_tensors="pt", padding=True, add_special_tokens=True)
        enc = {k: enc[k].to(device) for k in ("input_ids", "attention_mask")}   # drop e.g. token_type_ids
        kw = dict(max_new_tokens=C.LLM["max_new_tokens"], pad_token_id=tok.pad_token_id,
                  eos_token_id=tok.eos_token_id, stop_strings=["\n"], tokenizer=tok)
        if temperature > 0:
            kw.update(do_sample=True, temperature=temperature, top_k=0, top_p=1.0)
        else:
            kw.update(do_sample=False, temperature=None, top_k=None, top_p=None)
        out = model.generate(**enc, **kw)
        new_tokens = out[:, enc["input_ids"].shape[1]:]
        texts = tok.batch_decode(new_tokens, skip_special_tokens=True)
        return [t.split("\n")[0].strip() for t in texts]

    def translate_fn(texts, temperature):
        return translate_in_batches(texts, lambda b: gen_batch(b, temperature), max(8, C.EVAL_BATCH_SIZE // 2))

    return translate_fn


# --------------------------------------------------------------------------- #

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--family", choices=["llama", "gemma"], required=True)
    ap.add_argument("--mode", choices=["zeroshot", "fewshot", "lora"], default="lora")
    ap.add_argument("--model_id", default=None, help="override the Hugging Face model id from config.py")
    ap.add_argument("--n_train", type=int, default=C.N_TRAIN_LLM)
    ap.add_argument("--epochs", type=int, default=C.LLM["epochs"])
    ap.add_argument("--bs", type=int, default=None, help="micro-batch size (default: per family, see config.LLM_BATCH)")
    ap.add_argument("--grad_accum", type=int, default=None)
    ap.add_argument("--load_4bit", action="store_true", help="(experimental) load the base model in 4-bit NF4")
    ap.add_argument("--skip_temp", action="store_true")
    ap.add_argument("--eval_only", action="store_true", help="lora mode: load the saved adapter instead of training")
    args = ap.parse_args()

    args.bs = args.bs or C.LLM_BATCH[args.family][0]
    args.grad_accum = args.grad_accum or C.LLM_BATCH[args.family][1]
    name = f"{args.family}_{args.mode}"
    if args.mode == "lora":
        check_peft_torchao()
    model_id = args.model_id or C.LLM_IDS[args.family]
    set_seed()
    device = get_device()
    print("device:", device, "| run:", name)

    model, tok, dtype = load_model(model_id, args.family, device, args.load_4bit)
    base_total, _ = count_params(model)

    train_time, n_train, train_peak, trainable = 0.0, 0, 0.0, 0
    total = base_total
    if args.mode == "zeroshot":
        prompt_fn, adaptation = zero_shot_prompt, "Zero-shot prompting"
    elif args.mode == "fewshot":
        shots = pick_shots(load_split("train"), C.LLM["n_shots"])
        prompt_fn, adaptation = make_few_shot_prompt(shots), f"{len(shots)}-shot prompting"
    else:
        prompt_fn, adaptation = zero_shot_prompt, f"LoRA fine-tuning (r={C.LLM['lora_r']})"
        if args.eval_only:
            from peft import PeftModel
            model = PeftModel.from_pretrained(model, C.CKPT_DIR / name)
        else:
            model = lora_finetune(model, tok, dtype, args, device, name)
        tlog = load_json(C.RESULTS_DIR / f"{name}_train_log.json")
        train_time, n_train = tlog["train_time_s"], tlog["n_train_used"]
        train_peak, total, trainable = tlog["train_peak_gpu_mem_gb"], tlog["total_params"], tlog["trainable_params"]
        if not args.load_4bit:
            model = model.merge_and_unload()               # fuse adapters -> same inference cost as the base model

    meta = {
        "family": DISPLAY[args.family], "architecture": "Decoder-only", "adaptation": adaptation, "pretrained": True,
        "model_id": model_id, "tokenizer": f"{type(tok).__name__} ({len(tok)} tokens)",
        "total_params": total, "trainable_params": trainable, "n_train": n_train,
        "train_time_s": train_time, "train_peak_gpu_mem_gb": train_peak, "dtype": str(dtype),
    }
    evaluate_model(name, make_translate_fn(model, tok, device, prompt_fn), meta, skip_temp=args.skip_temp)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/tokenizer_stats.py
"""Tokenisation efficiency of each model on the test set (supports the tokenization discussion).

    python -m src.tokenizer_stats

Reports, for every tokenizer, the average number of sub-word tokens per Hindi word (reference
translations) and per English word (sources). A higher "tokens / Hindi word" means the model
needs more positions - more compute, more memory and longer generation - for the same sentence.
Tokenizers that cannot be downloaded (e.g. gated repos without a token) are skipped.
"""
from . import config as C
from .common import load_split, save_json


def avg_tokens_per_word(encode, texts):
    n_tok = sum(len(encode(t)) for t in texts)
    n_words = sum(len(t.split()) for t in texts)
    return round(n_tok / max(n_words, 1), 2)


def main():
    test = load_split("test")
    src = [r["src"] for r in test]
    tgt = [r["tgt"] for r in test]
    stats = {}

    spm_model = C.CKPT_DIR / "transformer_spm.model"
    if spm_model.exists():
        import sentencepiece as spm
        sp = spm.SentencePieceProcessor(model_file=str(spm_model))
        stats["Transformer (SentencePiece BPE, joint)"] = {
            "vocab_size": sp.get_piece_size(),
            "tokens_per_hi_word": avg_tokens_per_word(sp.encode, tgt),
            "tokens_per_en_word": avg_tokens_per_word(sp.encode, src)}

    try:
        from transformers import AutoTokenizer
    except ImportError:
        AutoTokenizer = None
        print("transformers not installed - skipping the Hugging Face tokenizers")
    hf = {"mT5": C.MT5["model_id"], "LLaMA": C.LLM_IDS["llama"], "Gemma": C.LLM_IDS["gemma"]} if AutoTokenizer else {}
    for name, model_id in hf.items():
        try:
            tok = AutoTokenizer.from_pretrained(model_id)
            enc = lambda t, tok=tok: tok(t, add_special_tokens=False)["input_ids"]
            stats[f"{name} ({model_id})"] = {
                "vocab_size": len(tok),
                "tokens_per_hi_word": avg_tokens_per_word(enc, tgt),
                "tokens_per_en_word": avg_tokens_per_word(enc, src)}
        except Exception as e:
            print(f"skipping {name}: {e}")

    save_json(C.RESULTS_DIR / "tokenizer_stats.json", stats)
    for k, v in stats.items():
        print(k, v)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/compare.py
"""Step 3 - aggregate everything in results/ into tables, plots and the README.

    python -m src.compare

Reads  results/<system>.json, *_probes.json, *_train_log.json, tokenizer_stats.json
Writes results/results_summary.csv, results_table.md, temperature_table.md, flags_table.md,
       probe_comparison.md, quality_comparison.png, compute_comparison.png,
       temperature_analysis.png, training_curves.png
and replaces the <!-- AUTO:...:START --> ... <!-- AUTO:...:END --> blocks inside README.md.
"""
import re

from . import config as C
from .common import load_json

PRIMARY = ["transformer", "mt5", "llama_lora", "gemma_lora"]     # the four systems of the assignment
R = C.RESULTS_DIR


# --------------------------------------------------------------------------- #

def load_results():
    out = {}
    for p in sorted(R.glob("*.json")):
        try:
            d = load_json(p)
        except Exception:
            continue
        if isinstance(d, dict) and "bleu" in d and "name" in d:
            out[d["name"]] = d
    return out


def label(d):
    return f"{d['family']} - {d['adaptation']}"


def short(d):
    return d["name"].replace("_", " ")


def md_table(headers, rows):
    lines = ["| " + " | ".join(headers) + " |", "|" + "|".join("---" for _ in headers) + "|"]
    for r in rows:
        lines.append("| " + " | ".join(str(c).replace("|", "\\|").replace("\n", " ") for c in r) + " |")
    return "\n".join(lines)


def fmt_min(sec):
    return "–" if not sec else f"{sec / 60:.1f}"


def main_rows(results, names):
    rows = []
    for n in names:
        d = results[n]
        rows.append([
            label(d), d.get("model_id", ""), d["architecture"], f"{d['total_params'] / 1e6:.1f}",
            f"{d['trainable_params'] / 1e6:.2f}" if d["trainable_params"] else "0 (frozen)",
            d["n_train"] or "–", fmt_min(d["train_time_s"]), f"{d['inference_time_s']:.1f}", d["ms_per_sentence"],
            d["bleu"], d["chrf"],
        ])
    return rows


MAIN_HEADERS = ["System", "Model", "Architecture", "Total params (M)", "Trainable params (M)", "Train pairs",
                "Train time (min)", "Inference (s, full test)", "ms / sentence", "BLEU", "chrF"]


# --------------------------------------------------------------------------- #
# Tables
# --------------------------------------------------------------------------- #

def build_tables(results):
    primary = [n for n in PRIMARY if n in results]
    extra = [n for n in results if n not in primary]
    tables = {}
    tables["main"] = md_table(MAIN_HEADERS, main_rows(results, primary)) if primary else "_No results yet - run the training scripts._"
    tables["extended"] = md_table(MAIN_HEADERS, main_rows(results, primary + extra)) if results else ""

    # temperature
    temps = sorted({t for d in results.values() for t in d.get("temperature_analysis", {})}, key=float)
    rows = []
    for n in primary + extra:
        ta = results[n].get("temperature_analysis", {})
        if ta:
            rows.append([label(results[n])] + [f"{ta[t]['bleu']} / {ta[t]['chrf']}" if t in ta else "–" for t in temps])
    tables["temperature"] = md_table(["System"] + [f"T={t}" for t in temps], rows) if rows else "_No temperature results yet._"
    tables["temperature_note"] = (f"Cells show BLEU / chrF on the first {next(iter(results.values())).get('temperature_analysis', {}).get('0.0', {}).get('n', C.N_TEMP_TEST) if results else C.N_TEMP_TEST} "
                                  "test sentences. T=0.0 is greedy decoding; T>0 is pure temperature sampling (no top-k/top-p), one seed.")

    # heuristic flags
    kinds = ["empty", "copied_source", "length_outlier", "repetition", "number_mismatch"]
    rows = [[label(results[n])] + [f"{results[n]['flags']['rate_pct'][k]}%" for k in kinds] for n in primary + extra]
    tables["flags"] = md_table(["System"] + kinds, rows) if rows else ""

    # tokenizer statistics
    tk = R / "tokenizer_stats.json"
    if tk.exists():
        t = load_json(tk)
        rows = [[k, v["vocab_size"], v["tokens_per_hi_word"], v["tokens_per_en_word"]] for k, v in t.items()]
        tables["tokenizer"] = md_table(["Tokenizer", "Vocab size", "Tokens / Hindi word", "Tokens / English word"], rows)
    else:
        tables["tokenizer"] = "_Run `python -m src.tokenizer_stats` to fill this table._"

    # dataset statistics
    ds = R / "dataset_stats.json"
    if ds.exists():
        s = load_json(ds)
        rows = [[k, v["n_pairs"], v["src_words_mean"], v["tgt_words_mean"], v["src_words_max"], v["tgt_words_max"],
                 v["src_vocab_types"], v["tgt_vocab_types"]] for k, v in s["splits"].items()]
        tables["dataset"] = (md_table(["Split", "Sentence pairs", "EN words (mean)", "HI words (mean)", "EN words (max)",
                                       "HI words (max)", "EN word types", "HI word types"], rows)
                             + f"\n\nRaw official split sizes: {s['raw_split_sizes']}. "
                               f"Training sizes used per model: {s['model_train_sizes']}.")
    else:
        tables["dataset"] = "_Run `python -m src.data` and `python -m src.compare` to fill the dataset table._"

    # best model
    if primary:
        by_chrf = max(primary, key=lambda n: (results[n]["chrf"], results[n]["bleu"]))
        by_bleu = max(primary, key=lambda n: results[n]["bleu"])
        fastest = min(primary, key=lambda n: results[n]["ms_per_sentence"])
        b = results[by_chrf]
        txt = (f"**Best overall (highest chrF, ties broken by BLEU): {label(b)}** - BLEU {b['bleu']}, chrF {b['chrf']}. "
               f"Highest BLEU: {label(results[by_bleu])} ({results[by_bleu]['bleu']}). "
               f"Fastest inference: {label(results[fastest])} ({results[fastest]['ms_per_sentence']} ms/sentence).")
        tables["best"] = txt
    else:
        tables["best"] = "_Results not available yet._"
    return tables


def probe_table(results):
    names = [n for n in PRIMARY if n in results and (R / f"{n}_probes.json").exists()]
    if not names:
        return "_No probe outputs yet._"
    data = {n: load_json(R / f"{n}_probes.json") for n in names}
    rows = []
    for i, p in enumerate(data[names[0]]):
        rows.append([p["id"], p["category"], p["src"]] + [data[n][i]["hyp"] for n in names])
    return md_table(["#", "Category", "English source"] + [results[n]["family"] for n in names], rows)


# --------------------------------------------------------------------------- #
# Plots
# --------------------------------------------------------------------------- #

def make_plots(results):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    primary = [n for n in PRIMARY if n in results]
    if not primary:
        return
    names = [f"{results[n]['family']}\n{results[n]['adaptation'].split('(')[0].strip()}" for n in primary]
    colors = ["#4c72b0", "#55a868", "#c44e52", "#8172b2"][: len(primary)]

    # quality
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
    for a, metric in zip(ax, ["bleu", "chrf"]):
        vals = [results[n][metric] for n in primary]
        bars = a.bar(names, vals, color=colors)
        a.bar_label(bars, fmt="%.1f")
        a.set_title(metric.upper() + " (higher is better)")
        a.tick_params(axis="x", labelsize=8)
    fig.tight_layout()
    fig.savefig(R / "quality_comparison.png", dpi=150)
    plt.close(fig)

    # compute
    fig, ax = plt.subplots(1, 3, figsize=(13, 3.8))
    spec = [("total_params", "Total parameters (M)", 1e6), ("train_time_s", "Training / fine-tuning time (min)", 60),
            ("ms_per_sentence", "Inference (ms / sentence)", 1)]
    for a, (key, title, div) in zip(ax, spec):
        vals = [results[n][key] / div for n in primary]
        bars = a.bar(names, vals, color=colors)
        a.bar_label(bars, fmt="%.1f")
        a.set_title(title + (" - log scale" if key == "total_params" else ""))
        if key == "total_params":
            a.set_yscale("log")                      # parameters span ~2 orders of magnitude
        a.tick_params(axis="x", labelsize=8)
    fig.tight_layout()
    fig.savefig(R / "compute_comparison.png", dpi=150)
    plt.close(fig)

    # temperature
    with_t = [n for n in results if results[n].get("temperature_analysis")]
    if with_t:
        fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
        for n in with_t:
            ta = results[n]["temperature_analysis"]
            ts = sorted(ta, key=float)
            for a, metric in zip(ax, ["bleu", "chrf"]):
                a.plot([float(t) for t in ts], [ta[t][metric] for t in ts], marker="o", label=short(results[n]))
        for a, metric in zip(ax, ["BLEU", "chrF"]):
            a.set_xlabel("sampling temperature (0 = greedy)")
            a.set_ylabel(metric)
            a.set_title(f"{metric} vs temperature")
        ax[1].legend(fontsize=7)
        fig.tight_layout()
        fig.savefig(R / "temperature_analysis.png", dpi=150)
        plt.close(fig)

    # training curves
    panels = []
    for n, title in [("transformer", "Transformer (scratch)"), ("mt5", "mT5 fine-tuning")]:
        p = R / f"{n}_train_log.json"
        if p.exists() and load_json(p)["log"]:
            panels.append((title, load_json(p)["log"], "epoch"))
    for n in ["llama_lora", "gemma_lora"]:
        p = R / f"{n}_train_log.json"
        if p.exists() and load_json(p)["log"]:
            panels.append((f"{n} LoRA", load_json(p)["log"], "step"))
    if panels:
        fig, ax = plt.subplots(1, len(panels), figsize=(4.2 * len(panels), 3.4), squeeze=False)
        for a, (title, log, xkey) in zip(ax[0], panels):
            xs = [r.get(xkey, k + 1) for k, r in enumerate(log)]
            a.plot(xs, [r["train_loss"] for r in log], marker="o", label="train loss")
            if "val_loss" in log[0]:
                a.plot(xs, [r["val_loss"] for r in log], marker="s", label="val loss")
            a.set_title(title, fontsize=10)
            a.set_xlabel(xkey)
            a.legend(fontsize=8)
        fig.tight_layout()
        fig.savefig(R / "training_curves.png", dpi=150)
        plt.close(fig)


# --------------------------------------------------------------------------- #

def update_readme(tables):
    readme = C.ROOT / "README.md"
    if not readme.exists():
        return
    text = readme.read_text(encoding="utf-8")
    for key, content in tables.items():
        pat = re.compile(rf"(<!-- AUTO:{key}:START -->)(.*?)(<!-- AUTO:{key}:END -->)", re.S)
        text = pat.sub(lambda m: f"{m.group(1)}\n{content}\n{m.group(3)}", text)
    readme.write_text(text, encoding="utf-8")
    print("README.md updated")


def main():
    results = load_results()
    if not results:
        print("No result files found in results/. Run the model scripts first.")
        return
    tables = build_tables(results)
    tables["probes"] = probe_table(results)

    import pandas as pd
    cols = ["name", "family", "architecture", "adaptation", "model_id", "tokenizer", "total_params", "trainable_params",
            "n_train", "train_time_s", "train_peak_gpu_mem_gb", "inference_time_s", "ms_per_sentence",
            "inference_peak_gpu_mem_gb", "bleu", "chrf", "device"]
    pd.DataFrame([{c: d.get(c) for c in cols} for d in results.values()]).to_csv(R / "results_summary.csv", index=False)

    (R / "results_table.md").write_text(tables["extended"] + "\n", encoding="utf-8")
    (R / "temperature_table.md").write_text(tables["temperature"] + "\n\n" + tables["temperature_note"] + "\n", encoding="utf-8")
    (R / "flags_table.md").write_text(tables["flags"] + "\n", encoding="utf-8")
    (R / "probe_comparison.md").write_text(tables["probes"] + "\n", encoding="utf-8")
    make_plots(results)
    update_readme(tables)
    print(tables["main"])
    print()
    print(tables["best"])


if __name__ == "__main__":
    main()


In [ ]:
%%writefile README.md
# English → Hindi Machine Translation: Encoder–Decoder vs Decoder-only Models

**Course:** Foundations of Generative AI (ICT4403) — Assignment 3
**System title (for the submission form):** *EN→HI Neural Machine Translation Benchmark: Basic Transformer and mT5 (encoder–decoder) vs LLaMA and Gemma (decoder-only autoregressive translators)*

> Sections marked **✍️** need a short write-up from you after you have run the experiments (they depend on your numbers).
> Everything inside `<!-- AUTO:... -->` blocks is overwritten by `python -m src.compare`.

---

## 1. Problem statement and objective

Machine translation (MT) can be framed as conditional text generation: given an English sentence, generate the Hindi sentence token by token. Two architectural families dominate:

* **Encoder–decoder** models read the whole source with a bidirectional encoder and generate the target with a decoder that cross-attends to it (Transformer, mT5).
* **Decoder-only** models treat translation as continuing a prompt (`English: … Hindi:`) with a single causal Transformer (LLaMA, Gemma).

**Objective.** Build four English→Hindi systems on the *same* public parallel corpus, generate translations for the *same* test sentences, and compare them on translation quality (BLEU, chrF), computational cost (parameters, training / fine-tuning time, inference time, memory), sampling-temperature sensitivity, and responsible-AI concerns (bias, hallucination, cultural misrepresentation, sensitive content).

| # | System | Architecture | Adaptation |
|---|--------|--------------|------------|
| 1 | Basic Transformer (implemented here in PyTorch) | encoder–decoder | trained from scratch |
| 2 | `google/mt5-small` | encoder–decoder | full fine-tuning |
| 3 | `meta-llama/Llama-3.2-1B-Instruct` | decoder-only | LoRA fine-tuning (+ zero-/few-shot prompting baselines) |
| 4 | `google/gemma-3-1b-it` (optionally the smaller `google/gemma-3-270m`) | decoder-only | LoRA fine-tuning (+ zero-/few-shot prompting baselines) |

## 2. Dataset

* **Name:** IIT Bombay English–Hindi Parallel Corpus (v3.1), ≈1.66 M training pairs, 520 validation pairs, 2,507 test pairs.
* **Source link:** <https://www.cfilt.iitb.ac.in/iitb_parallel/> · Hugging Face mirror: <https://huggingface.co/datasets/cfilt/iitb-english-hindi>
* **Licence:** CC BY-NC 4.0 (non-commercial use; attribution required).
* **Citation:** Kunchukuttan, Mehta & Bhattacharyya (2018). *The IIT Bombay English-Hindi Parallel Corpus.* LREC 2018. <https://aclanthology.org/L18-1548>
* **Content:** a mix of sources (government, tourism, news, technical/Gnome-style localisation strings, etc.), so some training pairs are noisy; the validation and test sets are the official WAT benchmark sets.

### Preprocessing (`src/data.py`)

1. Unicode NFC normalisation and whitespace collapsing.
2. **Evaluation splits first:** official validation set kept (only empty / extreme-length pairs dropped); **500 test sentences are sampled at random (seed 42)** from the official test set and used for *every* model.
3. **Training pool:** random sample of the official training split, then filtered — 3–50 words per side, length ratio 0.4–2.5, Hindi side ≥ 60 % Devanagari letters, English side ≥ 80 % Latin letters, source ≠ target, exact duplicates removed, **any sentence that also appears in validation/test removed (leakage check)**. Result: a cleaned pool of 100 k pairs (`data/train.jsonl`).
4. **Nested subsets:** each model trains on a prefix of this pool (see §6), so smaller budgets are always subsets of larger ones.

<!-- AUTO:dataset:START -->
_Run `python -m src.data` and `python -m src.compare` to fill the dataset table._
<!-- AUTO:dataset:END -->

![length histogram](results/length_hist.png)

## 3. Models

| | Basic Transformer | mT5-small | LLaMA (Llama-3.2-1B-Instruct) | Gemma (Gemma-3: 1B-it or 270M) |
|---|---|---|---|---|
| Family | encoder–decoder | encoder–decoder | decoder-only | decoder-only |
| Parameters | ≈ 9.6 M (tied embeddings) | ≈ 300 M (most are embeddings) | ≈ 1.2 B | ≈ 1.0 B (1B) or ≈ 0.27 B (270M); a large share are embeddings |
| Pretraining | none | mC4, 101 languages (span corruption) | large multilingual web corpus; Hindi is an officially supported language | large web corpus; the 1B size is text-only and its multilingual coverage is weaker than the 4B+ Gemma sizes (the 270M size is smaller still) |
| Tokenizer | joint 16 k SentencePiece-BPE trained on the 100 k pairs | 250 k SentencePiece | ≈ 128 k byte-level BPE | ≈ 262 k SentencePiece |
| Adaptation | train all weights | train all weights (Adafactor) | LoRA r = 16 on all attention + MLP projections | LoRA r = 16 on all attention + MLP projections |
| Prompting baselines | – | – | zero-shot, 3-shot | zero-shot, 3-shot |

**Basic Transformer** (`src/train_transformer.py`): pre-LayerNorm encoder–decoder, 3 + 3 layers, d_model 256, 4 heads, FFN 1024, dropout 0.1, sinusoidal positions, tied input/output embeddings, label smoothing 0.1, AdamW with inverse-sqrt warm-up, mixed precision, early stopping on validation loss.
**mT5** (`src/finetune_mt5.py`): `translate English to Hindi: <src>` → `<tgt>`, Adafactor, linear warm-up/decay, fp32 on T4 (mT5 overflows in fp16) or bf16 on Ampere+.
**LLaMA / Gemma** (`src/llm_mt.py`): 1B checkpoints used as language-model initialisations (plain text prompt, no chat template), prompt `Translate English to Hindi.\n\nEnglish: … \nHindi:`; loss is computed on the Hindi tokens only; generation stops at EOS or newline. LoRA adapters are merged into the weights before inference so inference cost equals the base model's.

## 4. Repository layout

```
├── README.md
├── requirements.txt
├── notebooks/FGAI_A3_EN_HI_MT.ipynb   # end-to-end Colab notebook (runs the scripts below)
├── notebooks/FGAI_A3_Kaggle.ipynb     # same pipeline for Kaggle
├── notebooks/FGAI_A3_AllInOne.ipynb   # single-file notebook: writes all project files itself and runs everything (Kaggle/Colab)
├── src/
│   ├── config.py             # every hyper-parameter / size (env-var overridable)
│   ├── data.py               # download, clean, split, describe the corpus
│   ├── common.py             # metrics, timing, heuristic flags, shared evaluation harness
│   ├── probes.py             # hand-written probe sentences for the ethics analysis
│   ├── train_transformer.py  # model 1
│   ├── finetune_mt5.py       # model 2
│   ├── llm_mt.py             # models 3 and 4 (zero-shot / few-shot / LoRA)
│   ├── tokenizer_stats.py    # tokens-per-word of every tokenizer
│   └── compare.py            # tables, plots, README auto-fill
├── results/                  # metrics (JSON/CSV), translations, plots, probe outputs
├── data/                     # generated by src/data.py (not committed)
└── checkpoints/              # model weights (not committed)
```

## 5. How to run

**Hardware.** Written for a single 16 GB GPU (free Colab T4 works; Ampere+ is faster). No GPU? Use the "tiny run" below to verify the pipeline, then run the full version on a GPU.

```bash
git clone https://github.com/<your-username>/<your-repo>.git && cd <your-repo>
pip install -r requirements.txt

# Llama and Gemma are gated: accept their licences on huggingface.co, then
export HF_TOKEN=hf_xxxxxxxxxxxxxxxx

python -m src.data                                          # 1. download + clean + split
python -m src.train_transformer                             # 2. model 1
python -m src.finetune_mt5                                  # 3. model 2
python -m src.llm_mt --family llama --mode lora             # 4. model 3 (main)
python -m src.llm_mt --family gemma --mode lora             # 5. model 4 (main)

# optional prompting baselines (no training, no temperature sweep)
for f in llama gemma; do for m in zeroshot fewshot; do python -m src.llm_mt --family $f --mode $m --skip_temp; done; done

python -m src.tokenizer_stats                               # 6. tokenizer efficiency
python -m src.compare                                       # 7. tables + plots + README
```

Or open `notebooks/FGAI_A3_EN_HI_MT.ipynb` in Colab (Runtime → T4 GPU) and run all cells.

**Tiny run (pipeline check, a few minutes):**

```bash
POOL_SIZE=2000 N_TEST=50 N_TEMP_TEST=20 N_TRAIN_SCRATCH=2000 N_TRAIN_MT5=1000 N_TRAIN_LLM=500 \
SCRATCH_EPOCHS=2 MT5_EPOCHS=1 SCRATCH_VOCAB=4000 python -m src.data   # then run the scripts above with the same variables
```

**Light mode (less GPU time / memory).** The assignment allows smaller variants and data subsets. Llama-3.2-1B is already the smallest Llama 3.x; for Gemma you can optionally use the 270M checkpoint, and smaller subsets:

```bash
GEMMA_ID=google/gemma-3-270m N_TRAIN_SCRATCH=50000 SCRATCH_EPOCHS=10 N_TRAIN_MT5=20000 N_TRAIN_LLM=5000 N_TEMP_TEST=100 python -m src.llm_mt --family gemma --mode lora   # same variables for every script
```

The notebooks have a `LIGHT = True` switch that does exactly this. The results table always lists the exact model id and number of training pairs used by every system.

**Troubleshooting.** `ImportError: Found an incompatible version of torchao` (seen on Kaggle when running the LoRA stages): run `pip uninstall -y torchao` — it is not used by this project. `CUDA out of memory`: lower the micro-batch (`--bs`) or the sizes in `src/config.py`.

**Re-evaluating without retraining:** every training script accepts `--eval_only`. **Reduce memory:** lower `--bs` / raise `--grad_accum` for the LLMs. All settings live in `src/config.py` and can be overridden with environment variables of the same name.

**Dependencies:** Python ≥ 3.10, PyTorch ≥ 2.3, `transformers` 4.46–4.x, `peft`, `datasets`, `sentencepiece`, `sacrebleu`, `pandas`, `matplotlib` (see `requirements.txt`).

## 6. Experimental settings (what is and is not held constant)

| Held constant for all four systems | Differs by design |
|---|---|
| language pair, corpus, cleaning, random seed (42) | training budget: Transformer 100 k pairs, mT5 50 k, LLaMA/Gemma LoRA 10 k (compute-driven; all are prefixes of the same pool) |
| the same 500 test sentences | model size, pretraining, tokenizer |
| greedy decoding for the main table (T = 0) | adaptation method (scratch / full FT / LoRA) |
| length-sorted batching, same metric code | numeric precision (fp16/bf16/fp32 chosen per model for stability) |
| BLEU + chrF via sacreBLEU (default 13a tokenisation, chrF char-6) | |

For a **controlled equal-data comparison**, rerun with `N_TRAIN_SCRATCH=10000 N_TRAIN_MT5=10000 N_TRAIN_LLM=10000`.
Inference time is wall-clock for the full test set with batched generation (longest-first). Training time is wall-clock of the optimisation loop only (tokenizer training for the scratch model is reported separately in the JSON).

## 7. Results

### 7.1 Main comparison (greedy decoding, 500 test sentences)

<!-- AUTO:main:START -->
_No results yet - run the training scripts, then `python -m src.compare`._
<!-- AUTO:main:END -->

<!-- AUTO:best:START -->
_Results not available yet._
<!-- AUTO:best:END -->

![quality](results/quality_comparison.png)
![compute](results/compute_comparison.png)
![training curves](results/training_curves.png)

### 7.2 All runs including prompting baselines

<!-- AUTO:extended:START -->
_No results yet._
<!-- AUTO:extended:END -->

### 7.3 Tokenization efficiency

<!-- AUTO:tokenizer:START -->
_Run `python -m src.tokenizer_stats` to fill this table._
<!-- AUTO:tokenizer:END -->

## 8. Model comparison and evaluation

Read §7 together with the table in §3. The four things the assignment asks you to consider:

1. **Model size.** Compare BLEU/chrF with the *Total params* and *Trainable params* columns. LoRA trains only a tiny fraction of the weights of a model that is several times larger than mT5-small, so "parameters" has two meanings (what is stored vs. what is learned).
2. **Pretraining.** The from-scratch Transformer has seen only the training pairs; mT5, LLaMA and Gemma bring knowledge from large multilingual pretraining. Check how much of any gap is explained by Hindi being (or not being) well covered in each model's pretraining data.
3. **Tokenization.** Use the tokenizer table: more tokens per Hindi word means longer sequences, more compute per sentence, slower generation and less content fitting in the same context. Relate it to the inference-time and memory columns.
4. **Adaptation method.** Full training vs. full fine-tuning vs. LoRA vs. prompting only (§7.2). Compare the zero-/few-shot rows with the LoRA rows to see how much task-specific fine-tuning adds for each decoder-only model.

**Encoder–decoder vs decoder-only (cost view).** Encoder–decoder models encode the source once and cross-attend to it, while decoder-only models re-process the prompt (including any few-shot examples) and carry it in the KV-cache during generation; decoder-only models here are also much larger. Use the *Inference*, *ms / sentence*, memory and *Train time* columns to quantify this on your hardware.

> **✍️ Write 5–8 sentences** stating, from your tables: which architecture family scored higher, by how much, at what computational cost, and which of the four factors above best explains the ranking.

## 9. Sampling-temperature analysis

Temperature *T* rescales the logits before the softmax, p(token) ∝ exp(z / T). *T → 0* is greedy decoding; *T = 1* samples from the model's own distribution; larger *T* flattens it, giving more diverse but riskier output. For translation, where one faithful output is wanted, quality usually falls as *T* rises — the question is how fast it falls for each model and whether hallucination-type flags increase.

<!-- AUTO:temperature:START -->
_No temperature results yet._
<!-- AUTO:temperature:END -->

<!-- AUTO:temperature_note:START -->
<!-- AUTO:temperature_note:END -->

![temperature](results/temperature_analysis.png)

> **✍️ Write 3–5 sentences**: how do BLEU and chrF change from T = 0 to T = 1 for each model? Which model is most robust to sampling and why (model size, calibration of its output distribution, fine-tuning)? Which temperature would you deploy?

## 10. Responsible and ethical AI analysis

**How the evidence was produced.** (a) 26 hand-written probe sentences (`src/probes.py`) covering gender bias, cultural content, idioms, sensitive content, numbers/entities and degenerate input are translated by every model (`results/probe_comparison.md`); (b) automatic red-flag heuristics are computed on the 500 test translations (below); (c) flagged examples are saved in `results/<model>_flag_examples.json` for manual review by a Hindi speaker.

<!-- AUTO:flags:START -->
_No flag statistics yet._
<!-- AUTO:flags:END -->

*Heuristics, not ground truth:* `empty` = no output; `copied_source` = output mostly Latin script (left untranslated); `length_outlier` = <0.5× or >2× the reference length (omission or over-generation); `repetition` = a word 3-gram repeated ≥ 3 times (degenerate looping); `number_mismatch` = a number in the source is missing from the output.

### 10.1 Concerns to examine in the outputs

| Concern | What to look for | Probe category |
|---|---|---|
| **Bias** | Hindi marks gender on verbs and adjectives, so the model must choose. Does *doctor/engineer/manager* default to masculine and *nurse* to feminine even when the English says otherwise? Are names or religions associated with stereotyped roles? | `gender_bias`, `cultural` (#8) |
| **Hallucinated information** | Content not in the source (invented names, dates, extra clauses), especially for very short or meaningless input; wrong numbers/dates/dosages; repetition loops; high-temperature drift. | `robustness`, `facts_numbers`, `number_mismatch`, `length_outlier` |
| **Cultural misrepresentation** | Literal translation of idioms; wrong kinship terms (e.g. *mother's brother's wife* → *māmī*); loss of honorifics or religious/festival terminology; transliteration instead of translation or vice-versa. | `cultural`, `idiom` |
| **Inappropriate translation of sensitive content** | Changes in meaning for medical, legal or violence-related text; trivialising or sensationalising wording; mistranslating terms about religion, disease or abuse; inconsistent register. | `sensitive` |

> **✍️ Add 3–6 concrete examples** from `results/probe_comparison.md` and the flag-example files (source → output → what is wrong → which model), and state which model was best/worst on each concern.

<!-- AUTO:probes:START -->
_No probe outputs yet._
<!-- AUTO:probes:END -->

### 10.2 Measures to reduce these concerns and use MT responsibly

* **Human in the loop for high-stakes text.** Medical, legal, safety and news content must be reviewed by a bilingual human before use; label machine output clearly as machine-translated.
* **Better data.** Audit and re-balance training data for gendered occupations and stereotyped associations; add parallel data with gender-neutral and both-gender variants; filter noisy pairs (as done here); prefer domain-specific, licensed data for the target use.
* **Gender handling at inference.** When the English is gender-ambiguous, offer both Hindi variants or ask for the gender instead of silently choosing; evaluate with gender-contrastive test sets.
* **Quality estimation and verification.** Use reference-free quality estimation / confidence scores to route low-confidence outputs to humans; automatically verify numbers, dates, units and named entities; use back-translation consistency checks; prefer greedy/beam decoding (low temperature) for production.
* **Controlled terminology.** Glossaries and constrained decoding for medical/legal terms, honorifics and festival/religious vocabulary.
* **Red-teaming and evaluation beyond BLEU.** Test with native speakers and challenge sets (idioms, kinship, code-mixing, dialects, sensitive topics); report per-category errors, not just a corpus score.
* **Guardrails and transparency.** Refuse or warn on unsafe/abusive content, avoid silently "softening" or "sharpening" meaning, publish model/data cards and known limitations, and respect the dataset licence (CC BY-NC 4.0 → non-commercial use only).
* **Privacy.** Do not send personal or confidential text to third-party translation services; filter personal data from training corpora.

## 11. Limitations

* Only a **subset** of the corpus is used and the models are small (≈ 0.01–1.2 B parameters); conclusions may not transfer to larger models or full-data training.
* Training budgets differ between systems (§6); the equal-data rerun isolates this effect.
* One training run and one sampling seed per configuration — differences of a few tenths of BLEU are within noise.
* BLEU with the default tokenizer is a coarse metric for Hindi (rich morphology, free word order); chrF is more forgiving. Neither measures adequacy errors such as a wrong number or a flipped gender.
* The IITB corpus is noisy and domain-skewed; the references themselves are imperfect.
* The automatic flags in §10 are heuristics and produce false positives (e.g. legitimate English acronyms in Hindi text).
* Small Gemma-3 checkpoints (1B, 270M) are text-only with weaker multilingual coverage than larger Gemma sizes, so expect weaker Hindi than Llama; a larger checkpoint can be tried with `--model_id` (optionally `--load_4bit`, experimental).

## 12. Conclusion

<!-- AUTO:best:START -->
<!-- AUTO:best:END -->

> **✍️ Write 4–6 sentences** naming the best-performing model, the evidence (BLEU/chrF, cost), the trade-off between quality and compute, and your recommendation for responsible deployment.

## 13. References

* Kunchukuttan, A., Mehta, P., Bhattacharyya, P. (2018). The IIT Bombay English-Hindi Parallel Corpus. LREC.
* Vaswani, A. et al. (2017). Attention Is All You Need. NeurIPS.
* Xue, L. et al. (2021). mT5: A Massively Multilingual Pre-trained Text-to-Text Transformer. NAACL.
* Hu, E. et al. (2022). LoRA: Low-Rank Adaptation of Large Language Models. ICLR.
* Meta AI. The Llama 3 Herd of Models (2024); Llama 3.2 model card.
* Gemma Team, Google DeepMind. Gemma 3 Technical Report (2025).
* Post, M. (2018). A Call for Clarity in Reporting BLEU Scores. WMT. · Popović, M. (2015). chrF: character n-gram F-score for automatic MT evaluation. WMT.


## 4. Install dependencies

In [ ]:
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], capture_output=True, text=True)
print(r.stdout[-1200:], r.stderr[-1200:])
assert r.returncode == 0, "pip install failed"
# peft + the old torchao that Kaggle ships are incompatible (LoRA stages crash) - torchao is not needed, remove it
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], capture_output=True)
import importlib.metadata as md_
for p in ["torch", "transformers", "peft", "datasets", "sacrebleu", "sentencepiece"]:
    print(p, md_.version(p))
# NOTE: "dependency resolver" warnings about gradio / diffusers / cudf etc. are harmless.

## 5. Hugging Face token + access check (stops early with a clear message if something is missing)

In [ ]:
tok = None
try:
    from kaggle_secrets import UserSecretsClient
    tok = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass
if not tok:
    try:
        from google.colab import userdata
        tok = userdata.get("HF_TOKEN")
    except Exception:
        pass
tok = tok or os.environ.get("HF_TOKEN")
if not tok:
    from getpass import getpass
    tok = getpass("Paste your Hugging Face token (hf_...): ")
os.environ["HF_TOKEN"] = tok

from huggingface_hub import hf_hub_download
hf_hub_download("cfilt/iitb-english-hindi", "README.md", repo_type="dataset"); print("OK dataset reachable")
for repo_id in [LLAMA_ID, GEMMA_ID]:
    try:
        hf_hub_download(repo_id, "config.json"); print("OK access:", repo_id)
    except Exception as e:
        raise RuntimeError(f"No access to {repo_id}. Accept its licence on huggingface.co with the SAME account as the token. ({type(e).__name__})")

## 6. Helpers

In [ ]:
os.environ.update(OVERRIDES)
if TINY_RUN:
    os.environ.update(TINY)          # tiny sizes win over OVERRIDES (model ids stay)

FAILED = []
def run(cmd, required=True):
    """Run a shell command and stream its output into the cell."""
    print("\n>>", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                         env={**os.environ, "PYTHONUNBUFFERED": "1"})
    for line in p.stdout:
        print(line, end="")
    p.wait()
    if p.returncode != 0:
        FAILED.append(cmd)
        if required:
            raise RuntimeError(f"FAILED: {cmd}")
        print(f"FAILED: {cmd} (continuing)")

def stage(name, cmd, required=True):
    """Skip a stage whose result file already exists (lets you re-run after a disconnect)."""
    if os.path.exists(f"results/{name}.json"):
        print(f"SKIP {name}: results/{name}.json already exists")
        return
    run(cmd, required)

## 7. Data (downloads the IIT Bombay English-Hindi corpus, cleans and splits it)

In [ ]:
if os.path.exists("data/test.jsonl"):
    print("SKIP data: already prepared")
else:
    run("python -m src.data")

## 8. Model 1 - Basic Transformer (from scratch)

In [ ]:
stage('transformer', 'python -m src.train_transformer')

## 9. Model 2 - mT5-small (fine-tuning)

In [ ]:
stage('mt5', 'python -m src.finetune_mt5')

## 10. Model 3 - Llama-3.2-1B-Instruct (LoRA)

In [ ]:
stage('llama_lora', 'python -m src.llm_mt --family llama --mode lora', required=False)

## 11. Model 4 - Gemma (LoRA)

In [ ]:
stage('gemma_lora', 'python -m src.llm_mt --family gemma --mode lora', required=False)

## 12. Prompting baselines for the two LLMs (optional, no training)

In [ ]:
for fam in ["llama", "gemma"]:
    for mode in ["zeroshot", "fewshot"]:
        stage(f"{fam}_{mode}", f"python -m src.llm_mt --family {fam} --mode {mode} --skip_temp", required=False)

## 13. Tokenizer stats, comparison tables, plots, README

In [ ]:
run("python -m src.tokenizer_stats", required=False)
run("python -m src.compare")
print("\nFailed commands:", FAILED or "none")

In [ ]:
from IPython.display import Markdown, Image, display
display(Markdown(open("results/results_table.md").read()))
for f in ["quality_comparison", "compute_comparison", "temperature_analysis", "training_curves"]:
    if os.path.exists(f"results/{f}.png"):
        display(Image(f"results/{f}.png"))

## 14. Look at some translations

In [ ]:
import random
from src.common import read_jsonl
names = ["transformer", "mt5", "llama_lora", "gemma_lora"]
data = {n: read_jsonl(f"results/{n}_translations.jsonl") for n in names if os.path.exists(f"results/{n}_translations.jsonl")}
if data:
    first = next(iter(data.values()))
    for i in random.Random(0).sample(range(len(first)), min(6, len(first))):
        print("EN :", first[i]["src"]); print("REF:", first[i]["ref"])
        for n, rows in data.items():
            print(f"{n:12s}:", rows[i]["hyp"])
        print("-" * 80)

## 15. Bundle everything for download

In [ ]:
def add_tree(z, base, folder, skip=()):
    for root, dirs, fs in os.walk(folder):
        dirs[:] = [d for d in dirs if d not in skip and d != "__pycache__"]
        for f in fs:
            z.write(os.path.join(root, f), os.path.join(base, root, f))

with zipfile.ZipFile(ROOT / "results_bundle.zip", "w", zipfile.ZIP_DEFLATED) as z:      # results + auto-filled README
    add_tree(z, "", "results")
    z.write("README.md")
with zipfile.ZipFile(ROOT / "project_for_github.zip", "w", zipfile.ZIP_DEFLATED) as z:   # whole repo (no weights / raw data)
    for top in ["src", "results"]:
        add_tree(z, PROJECT.name, top)
    for f in ["README.md", "requirements.txt"]:
        z.write(f, f"{PROJECT.name}/{f}")
if CLEAN_CHECKPOINTS and not TINY_RUN:
    shutil.rmtree("checkpoints", ignore_errors=True)

print("Created:", ROOT / "results_bundle.zip", "and", ROOT / "project_for_github.zip")
if TINY_RUN:
    print("\n*** TINY RUN FINISHED - these are NOT real results. Set TINY_RUN = False and run again. ***")
if PLATFORM == "colab":
    from google.colab import files
    files.download(str(ROOT / "results_bundle.zip")); files.download(str(ROOT / "project_for_github.zip"))